# Clase 8 — RAG avanzado y transición a Tools

En C6 construimos un **retriever evaluable** con corpus, ground truth y métricas.  
En C7 lo convertimos en un **RAG end-to-end** con evidencia, trazabilidad y política de respuesta.

C8 toma ese sistema como baseline y avanza en **dos actos consecutivos**:

```text
ACTO 1 — mejorar y evaluar retrieval
mismo corpus + mismo ground truth de C6
        ↓
Dense
        ↓
BM25 + Hybrid + RRF
        ↓
Reranking
        ↓
métricas + diagnóstico + RAG Tournament
```

```text
ACTO 2 — convertir búsqueda en una capacidad seleccionable
interfaz estable de búsqueda
        ↓
descripción de capacidades
        ↓
Semantic / LLM Router
        ↓
aplicación al caso transversal de emails
        ↓
handoff hacia C9
```

La separación es intencional: primero respondemos **“¿recuperamos bien la evidencia?”** y recién después **“¿cuándo corresponde usar esa capacidad?”**.

> **Límite de C8:** todavía no implementamos Function Calling, Tool Registry ni Agent Loop. Eso comienza en C9.


## Mapa del laboratorio

Al finalizar deberías poder:

- recuperar el corpus y el golden dataset persistidos en C6;
- usar el dense retriever de C6 como baseline;
- distinguir **dense retrieval** de **lexical/sparse retrieval**;
- usar BM25 como señal lexical complementaria;
- fusionar rankings con **Reciprocal Rank Fusion (RRF)** sin sumar scores incompatibles;
- entender qué es un **candidate set** y por qué condiciona al reranker;
- explicar la diferencia entre **Bi-Encoder** y **Cross-Encoder**;
- comparar `dense`, `hybrid + RRF` y `hybrid + reranking` sobre el mismo ground truth;
- conservar **Precision@k, Recall@k, Hit Rate@k, MRR y latencia**;
- diagnosticar si un problema es de cobertura, ruido, orden o ausencia de evidencia;
- observar cómo cambia la calidad de los primeros puestos del contexto;
- justificar una estrategia en el **RAG Tournament** usando evidencia del benchmark;
- encapsular retrieval detrás de una interfaz estable como `search_course()`;
- describir esa búsqueda como una capacidad seleccionable sin convertirla todavía en una Tool ejecutable;
- evaluar por separado un **golden de routing**;
- distinguir con claridad **Router ≠ Agent**;
- aplicar el patrón al caso transversal de emails y dejar el handoff para C9.


### Patrón de trabajo de esta notebook

Mantenemos el patrón acumulativo de las clases anteriores:

1. **reutilizar** componentes ya construidos;
2. **aislar** una decisión técnica;
3. **comparar** contra un baseline;
4. **diagnosticar** con métricas antes de elegir arquitectura;
5. **persistir** después de entender el comportamiento;
6. dejar un **handoff explícito** para la clase siguiente.

Durante el primer acto mantenemos fijos corpus, consultas y ground truth. Así podemos atribuir los cambios de calidad a la estrategia de retrieval/ranking y no a que cambiamos el examen.


### Caso transversal — emails, reservado para el segundo acto

El proyecto mantiene como caso transversal un **agente de gestión asistida de emails**. C7 dejó persistidos:

```text
data/mock_gmail/threads.json
data/mock_gmail/knowledge_base.md
src/ai_agent_course/support_case.py
```

En C8 no usamos esos mails para reemplazar el benchmark técnico. Primero cerramos la comparación de retrieval sobre el dataset heredado de C6; después reutilizamos la arquitectura resultante sobre la KB de soporte y estudiamos **routing**.

De esta manera no mezclamos dos objetivos distintos:

1. **calidad de retrieval**;
2. **selección de capacidad**.


In [29]:
# Prepara rutas y utilidades del mismo proyecto incremental.
# C8 sigue trabajando sobre el proyecto construido en clases anteriores:
# no crea un proyecto nuevo, sino que agrega capacidades al mismo paquete.
from pathlib import Path
import importlib
import importlib.util
import json
import os
import re
import statistics
import subprocess
import sys
import textwrap
import time

PROJECT_DIR = Path("ai_agent_project").resolve()
SRC_DIR = PROJECT_DIR / "src" / "ai_agent_course"
DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
TESTS_DIR = PROJECT_DIR / "tests"

# Creamos las carpetas de forma idempotente: si ya existen, se reutilizan.
for directory in (SRC_DIR, DATA_DIR, ARTIFACTS_DIR, TESTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# __init__.py permite importar ai_agent_course como paquete local.
(SRC_DIR / "__init__.py").touch()

# Agregamos src/ al path para importar los módulos que escribimos con %%writefile.
if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))


def write_file(path: Path, content: str) -> None:
    """Guarda texto dentro del proyecto y asegura que exista la carpeta destino.

    Args:
        path: Ruta del archivo que queremos crear o actualizar.
        content: Contenido textual que se escribirá en UTF-8.

    Returns:
        None. La función produce el archivo como efecto lateral.

    Se usa al final para persistir reportes y artefactos reproducibles.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).strip() + "\n", encoding="utf-8")
    print("✓", path.relative_to(PROJECT_DIR))


def show_table(rows, columns=None):
    """Muestra una colección de filas en formato tabular para inspección.

    Args:
        rows: Registros que pandas puede convertir a DataFrame.
        columns: Columnas y orden a mostrar; si es None, se muestran todas.

    Returns:
        None. Presenta la tabla en notebook o imprime filas si falta pandas.

    El fallback mantiene el laboratorio usable en entornos mínimos sin ocultar
    otros errores distintos de la ausencia de pandas.
    """
    try:
        import pandas as pd
        frame = pd.DataFrame(rows)
        if columns:
            frame = frame[columns]
        display(frame)
    except ImportError:
        for row in rows:
            print(row)


# Smoke mode privilegia ejecución reproducible/offline; fuera de ese modo
# intentaremos usar componentes reales cuando las dependencias estén disponibles.
SMOKE_MODE = False

print("Proyecto incremental:", PROJECT_DIR)
print("Modo:", "smoke / offline" if SMOKE_MODE else "laboratorio real")


Proyecto incremental: /Users/gasparcartasso/Desktop/AI Developer/ai_agent_project
Modo: laboratorio real


### Checkpoint de continuidad

C8 necesita dos handoffs distintos:

- **C6** aporta corpus, golden dataset y métricas de retrieval;
- **C7** aporta el RAG end-to-end, los fixtures de emails y LangChain ya trabajado.

Por eso **LangChain no es una variable nueva de C8**. El primer bloque se concentra en retrieval híbrido, ranking y evaluación; el segundo, en selección de capacidad y routing.


In [30]:
# Verificamos el contrato de continuidad antes de ejecutar C8.
# Si falta un módulo o reporte heredado, fallamos temprano con un mensaje claro.
required_previous_files = [
    SRC_DIR / "embeddings.py",
    SRC_DIR / "retrieval.py",
    SRC_DIR / "rag.py",
    SRC_DIR / "generation.py",
    SRC_DIR / "support_case.py",
    ARTIFACTS_DIR / "class06_report.json",
    ARTIFACTS_DIR / "class07_report.json",
]

# Registramos disponibilidad para mostrar exactamente qué dependencia falta.
continuity_status = [
    {
        "path": str(path.relative_to(PROJECT_DIR)),
        "available": path.exists(),
    }
    for path in required_previous_files
]

for item in continuity_status:
    print("✓" if item["available"] else "·", item["path"])

missing_previous = [
    item["path"]
    for item in continuity_status
    if not item["available"]
]

# Evitamos continuar con estado parcial: C8 compara contra artefactos históricos.
if missing_previous:
    raise RuntimeError(
        "Ejecutá las clases previas antes de continuar. Faltan: "
        + ", ".join(missing_previous)
    )

print("✓ Continuidad con C6 y C7 verificada")


✓ src/ai_agent_course/embeddings.py
✓ src/ai_agent_course/retrieval.py
✓ src/ai_agent_course/rag.py
✓ src/ai_agent_course/generation.py
✓ src/ai_agent_course/support_case.py
✓ artifacts/class06_report.json
✓ artifacts/class07_report.json
✓ Continuidad con C6 y C7 verificada


## 1. Dependencias y baseline heredado

Las únicas dependencias nuevas relevantes para el núcleo de C8 son:

- `rank-bm25`: implementación de BM25;
- `sentence-transformers`: solo si queremos ejecutar un Cross-Encoder real.

El laboratorio puede correr en **smoke mode** sin descargar el Cross-Encoder. En ese caso usamos un fallback explícito para comprobar el contrato, pero **no fingimos que ese fallback es un Cross-Encoder**.


In [31]:
# Las instalaciones son opt-in para no modificar silenciosamente el entorno del alumno.
RUN_INSTALLS = os.getenv("RUN_INSTALLS", "0") == "1"
print("RUN_INSTALLS:", RUN_INSTALLS)
PACKAGES = [
    "rank-bm25==0.2.2",
    "sentence-transformers>=3.0",
]

# RUN_INSTALLS=1 prepara un entorno vacío; por defecto solo mostramos el comando.
if RUN_INSTALLS:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        *PACKAGES,
    ])
else:
    print("Dependencias opcionales:")
    print("python -m pip install " + " ".join(PACKAGES))

# Detectamos capacidades opcionales para elegir fallbacks de forma explícita.
HAS_QDRANT = importlib.util.find_spec("qdrant_client") is not None
HAS_SENTENCE_TRANSFORMERS = (
    importlib.util.find_spec("sentence_transformers") is not None
)

print({
    "qdrant_client": HAS_QDRANT,
    "sentence_transformers": HAS_SENTENCE_TRANSFORMERS,
})


RUN_INSTALLS: False
Dependencias opcionales:
python -m pip install rank-bm25==0.2.2 sentence-transformers>=3.0
{'qdrant_client': True, 'sentence_transformers': True}


In [32]:
# Recuperamos dos handoffs distintos:
# - C6: benchmark técnico de retrieval.
# - C7: caso transversal de emails.

from ai_agent_course.embeddings import build_embedding_provider
from ai_agent_course.retrieval import build_retriever
from ai_agent_course.chunking import document_chunks
from ai_agent_course.support_case import load_support_case

# Cargamos los reportes como fuente de verdad: así C8 no redefine datasets
# ni configuración histórica y las comparaciones siguen siendo válidas.
class06_report = json.loads(
    (ARTIFACTS_DIR / "class06_report.json").read_text(encoding="utf-8")
)
class07_report = json.loads(
    (ARTIFACTS_DIR / "class07_report.json").read_text(encoding="utf-8")
)

# ------------------------------------------------------------------
# 1) BENCHMARK PRINCIPAL: reutilizar literalmente C6
# ------------------------------------------------------------------
CHUNKS = class06_report["handoff"]["chunks"]
RETRIEVAL_GOLDEN = class06_report["handoff"]["golden_dataset"]

# Conservamos el alias para simplificar algunas celdas posteriores.
GOLDEN_DATASET = RETRIEVAL_GOLDEN

# Separamos casos con evidencia de los controles negativos.
# Las métricas de ranking se calculan sobre positivos; los negativos se inspeccionan aparte.
positive_cases = [
    case
    for case in RETRIEVAL_GOLDEN
    if case["expects_evidence"]
]

no_evidence_cases = [
    case
    for case in RETRIEVAL_GOLDEN
    if not case["expects_evidence"]
]

# Mismo tipo de embedding usado en C6: si lo cambiáramos,
# ya no estaríamos comparando únicamente la estrategia de retrieval.
use_real_embeddings = class06_report["embedding"]["mode"] == "real"

if use_real_embeddings and not HAS_SENTENCE_TRANSFORMERS:
    raise RuntimeError(
        "C6 usó embeddings reales. Instalá sentence-transformers "
        "para conservar la misma configuración."
    )

embedding_provider = build_embedding_provider(
    use_real=use_real_embeddings
)

# Reconstruimos el baseline dense con los mismos embeddings de C6.
# Este índice será la referencia frente a Hybrid + RRF y Hybrid + Rerank.
dense_retriever = build_retriever(
    use_qdrant=False,
    path=DATA_DIR / "class08_dense_baseline",
    embedding_provider=embedding_provider,
)
dense_retriever.index(CHUNKS)

# ------------------------------------------------------------------
# 2) CASO TRANSVERSAL: fixture de mails heredado de C7
# ------------------------------------------------------------------
THREADS, KNOWLEDGE_BASE = load_support_case(PROJECT_DIR)

def thread_to_query(thread: dict) -> str:
    """Convierte un thread de email en una única consulta para retrieval.

    Args:
        thread: Diccionario con ``subject`` y una lista opcional de ``messages``.

    Returns:
        Texto formado por asunto + cuerpos de mensajes, listo para buscar.

    Esta transformación conserva suficiente contexto del hilo sin mezclar todavía
    routing, retrieval y generación en una sola función.
    """
    bodies = [
        message.get("body", "")
        for message in thread.get("messages", [])
    ]
    return " ".join(
        [thread.get("subject", ""), *bodies]
    ).strip()

SUPPORT_DOC_IDS = {
    "Carga de facturas": "SUP-FACTURAS",
    "Cambio de contraseña": "SUP-PASSWORD",
    "Email de validación": "SUP-VALIDACION",
    "Integraciones y API": "SUP-API",
}

def support_documents(knowledge_base: str) -> list[dict]:
    """Normaliza una KB Markdown al contrato de documentos usado por el curso.

    Args:
        knowledge_base: Texto Markdown donde cada ``##`` inicia un tema.

    Returns:
        Lista de documentos con id, texto, fuente y metadata.

    El objetivo es que la KB de soporte pueda atravesar el mismo pipeline de
    chunking y retrieval que cualquier otro corpus del proyecto.
    """
    sections = re.split(r"\n## ", knowledge_base)
    documents = []

    # Cada sección se vuelve un documento lógico antes de aplicar chunking.
    for section in sections:
        clean = section.strip()
        if not clean or clean.startswith("#"):
            continue

        lines = clean.splitlines()
        topic = lines[0].strip()
        body = "\n".join(lines[1:]).strip()

        # Preferimos ids estables para los temas conocidos porque el golden
        # evalúa documentos por identidad, no por el texto de su título.
        document_id = SUPPORT_DOC_IDS.get(
            topic,
            "SUP-" + re.sub(
                r"[^A-Z0-9]+",
                "-",
                topic.upper(),
            ).strip("-"),
        )

        documents.append({
            "document_id": document_id,
            "source": "support_knowledge_base.md",
            "page": 1,
            "text": f"{topic}\n{body}",
            "metadata": {
                "topic": topic,
                "department": "support",
                "active": True,
            },
        })

    return documents

# Normalizamos primero a documentos y recién después dividimos en chunks.
# Así la trazabilidad document_id -> chunk_id queda igual que en C6/C7.
SUPPORT_DOCUMENTS = support_documents(KNOWLEDGE_BASE)
SUPPORT_CHUNKS = document_chunks(
    SUPPORT_DOCUMENTS,
    target_words=45,
    overlap_words=5,
)

# Ground truth mínimo del caso transversal: cada hilo apunta al documento
# de soporte que debería aportar la evidencia principal.
EXPECTED_DOC_BY_THREAD = {
    "thr_001": "SUP-FACTURAS",
    "thr_002": "SUP-PASSWORD",
    "thr_003": "SUP-API",
}

EMAIL_CASES = [
    {
        "case_id": thread["thread_id"],
        "thread_id": thread["thread_id"],
        "subject": thread["subject"],
        "query": thread_to_query(thread),
        "relevant_document_ids": [
            EXPECTED_DOC_BY_THREAD[thread["thread_id"]]
        ],
    }
    for thread in THREADS
    if thread["thread_id"] in EXPECTED_DOC_BY_THREAD
]

print({
    "benchmark": {
        "chunks": len(CHUNKS),
        "golden_cases": len(RETRIEVAL_GOLDEN),
        "positive_cases": len(positive_cases),
        "negative_cases": len(no_evidence_cases),
    },
    "email_case": {
        "threads": len(THREADS),
        "support_chunks": len(SUPPORT_CHUNKS),
    },
    "embedding_model": embedding_provider.model_name,
    "c7_generation": class07_report["generation"]["provider"],
})

show_table(
    RETRIEVAL_GOLDEN,
    [
        "case_id",
        "category",
        "query",
        "relevant_document_ids",
        "expects_evidence",
    ],
)


{'benchmark': {'chunks': 10, 'golden_cases': 6, 'positive_cases': 5, 'negative_cases': 1}, 'email_case': {'threads': 3, 'support_chunks': 4}, 'embedding_model': 'deterministic-semantic-smoke-v3', 'c7_generation': 'extractive-offline-fallback'}


,case_id,category,query,relevant_document_ids,expects_evidence
0,VAC-01,vacaciones,¿con cuánta anticipación pido vacaciones?,"[POL-VAC-001, FAQ-RRHH-005]",True
1,EXP-01,gastos,¿cuál es el plazo para presentar un reintegro?,[POL-EXP-002],True
2,SEC-01,seguridad,"expuse mi contraseña, ¿qué acciones debo tomar?","[POL-SEC-003, INC-SEC-006]",True
3,REM-01,trabajo remoto,¿cómo se registra el trabajo remoto?,[POL-REM-004],True
4,CAP-01,capacitación,¿qué necesito para pedir reintegro de una capa...,[POL-BEN-007],True
5,NONE-01,sin evidencia,¿la empresa ofrece estacionamiento gratuito?,[],False


### 1.1 Ground truth: qué estamos preservando

En C6 definimos un **golden dataset**: consultas para las que conocemos de antemano qué documentos deberían ser relevantes.

Ese conocimiento esperado es el **ground truth**.

En esta clase no lo reconstruimos ni lo adaptamos a la técnica nueva. Lo reutilizamos tal cual:

- `VAC-01` — vacaciones;
- `EXP-01` — gastos;
- `SEC-01` — seguridad;
- `REM-01` — trabajo remoto;
- `CAP-01` — capacitación;
- `NONE-01` — control negativo sin evidencia.

La razón es metodológica:

> Si cambiamos el ground truth al mismo tiempo que cambiamos retrieval, dejamos de tener una comparación controlada.

C8 agrega técnicas nuevas **contra el mismo examen**.


In [33]:
# Checkpoint: comprobamos que C8 usa exactamente el mismo golden de C6.
# Esta igualdad evita que una mejora aparente provenga de cambiar los casos de prueba.
print("Golden heredado de C6")
show_table(
    RETRIEVAL_GOLDEN,
    [
        "case_id",
        "category",
        "query",
        "relevant_document_ids",
        "expects_evidence",
    ],
)

assert RETRIEVAL_GOLDEN == class06_report["handoff"]["golden_dataset"]
print("✓ Ground truth de C6 reutilizado sin redefinir")


Golden heredado de C6


,case_id,category,query,relevant_document_ids,expects_evidence
0,VAC-01,vacaciones,¿con cuánta anticipación pido vacaciones?,"[POL-VAC-001, FAQ-RRHH-005]",True
1,EXP-01,gastos,¿cuál es el plazo para presentar un reintegro?,[POL-EXP-002],True
2,SEC-01,seguridad,"expuse mi contraseña, ¿qué acciones debo tomar?","[POL-SEC-003, INC-SEC-006]",True
3,REM-01,trabajo remoto,¿cómo se registra el trabajo remoto?,[POL-REM-004],True
4,CAP-01,capacitación,¿qué necesito para pedir reintegro de una capa...,[POL-BEN-007],True
5,NONE-01,sin evidencia,¿la empresa ofrece estacionamiento gratuito?,[],False


✓ Ground truth de C6 reutilizado sin redefinir


### 1.2 Baseline Dense

Antes de agregar señales nuevas, ejecutamos una consulta del golden con el retriever Dense heredado. Este ranking será el punto de comparación para el resto del bloque técnico.


In [34]:
# Baseline real: volvemos a una consulta del golden de C6.
baseline_case = positive_cases[0]
baseline_query = baseline_case["query"]

# Ejecutamos primero Dense sin ninguna mejora avanzada.
# El ranking resultante sirve como punto de comparación para el resto de la clase.
baseline_results = dense_retriever.search(
    baseline_query,
    top_k=5,
)

print("CASO:", baseline_case["case_id"])
print("CATEGORÍA:", baseline_case["category"])
print("QUERY:", baseline_query)
print("GROUND TRUTH:", baseline_case["relevant_document_ids"])

# Mostramos documento, chunk y score para poder inspeccionar calidad y trazabilidad.
show_table([
    {
        "rank": rank,
        "document_id": result.document_id,
        "chunk_id": result.chunk_id,
        "score": round(float(result.score), 4),
        "preview": result.text[:120],
    }
    for rank, result in enumerate(
        baseline_results,
        start=1,
    )
])


CASO: VAC-01
CATEGORÍA: vacaciones
QUERY: ¿con cuánta anticipación pido vacaciones?
GROUND TRUTH: ['POL-VAC-001', 'FAQ-RRHH-005']


,rank,document_id,chunk_id,score,preview
0,1,FAQ-RRHH-005,FAQ-RRHH-005:recursive:000:8127fc62,0.8680,# Preguntas frecuentes de RRHH Para pedir vaca...
1,2,POL-VAC-001,POL-VAC-001:recursive:000:f05eeb6e,0.8642,Las solicitudes de vacaciones deben registrars...
2,3,POL-VAC-001,POL-VAC-001:recursive:001:2bc935bb,0.0061,cobertura operativa y las fechas solicitadas. ...
3,4,POL-EXP-002,POL-EXP-002:recursive:001:3ac2515d,0.0000,"ilegibles, duplicados o fuera de plazo. Cuando..."
4,5,POL-SEC-003,POL-SEC-003:recursive:001:b6ee93f8,0.0000,abrir inmediatamente un incidente de seguridad...


## 2. Dense vs. lexical retrieval: dos señales distintas

**Dense retrieval** representa query y documentos como vectores densos y busca cercanía semántica. Es fuerte para:

- paráfrasis;
- intención;
- expresiones distintas con significado parecido.

**Lexical / sparse retrieval** trabaja principalmente con términos presentes en el texto. Es fuerte para:

- códigos;
- siglas;
- nombres poco frecuentes;
- coincidencias exactas.

Ninguna señal domina siempre. **Hybrid Search** combina ambas porque sus errores suelen ser complementarios.


### 2.1 BM25: una señal lexical complementaria

BM25 premia coincidencias de términos considerando, entre otras cosas:

- frecuencia del término;
- rareza del término en el corpus;
- longitud del documento.

Una forma simplificada de leerlo es:

```text
BM25(d, q) ≈ Σ IDF(term)
              × función_de_frecuencia(term, d)
              × normalización_por_longitud(d)
```

`k1` controla cuánto crece el aporte de repetir un término y `b` cuánto pesa la normalización por longitud. **No necesitamos memorizar la ecuación** para usar la técnica correctamente.

La intuición sí importa:

> **BM25 protege términos exactos que una representación semántica puede diluir.**


In [35]:
# Preview mínimo antes de persistir el retriever lexical.
def preview_tokenize(text: str) -> list[str]:
    """Tokeniza texto de forma mínima para visualizar la señal léxica.

    Args:
        text: Consulta o documento en texto plano.

    Returns:
        Lista de tokens en minúsculas.

    Es solo un preview: el módulo persistido define la versión reutilizable.
    """
    return re.findall(r"\w+", text.lower())


preview_chunks = [
    {"chunk_id": "vac", "text": "vacaciones licencia anual anticipación"},
    {"chunk_id": "gas", "text": "gastos reintegro comprobante factura"},
    {"chunk_id": "seg", "text": "seguridad contraseña credenciales"},
]

# BM25 trabaja sobre tokens; no utiliza embeddings ni similitud semántica.
preview_docs = [
    preview_tokenize(chunk["text"])
    for chunk in preview_chunks
]

try:
    from rank_bm25 import BM25Okapi

    # rank-bm25 calcula un score por documento para los términos de la consulta.
    preview_model = BM25Okapi(preview_docs)
    preview_scores = preview_model.get_scores(
        preview_tokenize("reintegro comprobante")
    )
    preview_backend = "rank-bm25-okapi"
except ImportError:
    # Fallback visible para que el notebook sea ejecutable sin instalación.
    preview_scores = [0.0, 2.0, 0.0]
    preview_backend = "preview-fallback-explicit"

preview_rows = sorted(
    [
        {
            "chunk_id": chunk["chunk_id"],
            "score": round(float(score), 4),
        }
        for chunk, score in zip(preview_chunks, preview_scores, strict=True)
    ],
    key=lambda row: row["score"],
    reverse=True,
)

assert preview_rows[0]["chunk_id"] == "gas"

print("Backend:", preview_backend)
show_table(preview_rows)


Backend: rank-bm25-okapi


,chunk_id,score
0,gas,0.9815
1,vac,0.0000
2,seg,0.0000


### 2.2 Persistir BM25 y RRF

Vamos a guardar en un único módulo:

- tokenización;
- `BM25Retriever`;
- la estructura de candidatos fusionados;
- `rrf_fuse()`.

RRF combina **posiciones**, no scores. Eso evita el problema de sumar directamente similitud vectorial y BM25, que viven en escalas distintas.


In [36]:
%%writefile ai_agent_project/src/ai_agent_course/ranking.py
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
import math
import re
from typing import Sequence

from .retrieval import SearchResult



def tokenize(text: str) -> list[str]:
    """Normaliza texto a tokens simples para el índice léxico.

    Args:
        text: Consulta o contenido de un chunk.

    Returns:
        Lista de tokens alfanuméricos en minúsculas.

    La misma normalización se aplica a documentos y queries para que BM25
    compare términos en un espacio léxico consistente.
    """
    return re.findall(r"\w+", text.lower())


@dataclass(frozen=True)
class LexicalResult:
    """Representa un chunk recuperado por BM25 junto con su score léxico."""

    chunk: dict
    score: float


@dataclass(frozen=True)
class FusedCandidate:
    """Candidato fusionado por RRF con trazabilidad de sus rankings de origen."""

    result: SearchResult
    rrf_score: float
    dense_rank: int | None
    lexical_rank: int | None


class BM25Retriever:
    """Retriever léxico basado en BM25 con fallback didáctico offline.

    Recibe chunks ya creados y devuelve coincidencias por términos. Su función
    arquitectónica es complementar al retriever dense con una señal distinta:
    nombres, siglas o palabras exactas que los embeddings pueden ordenar peor.
    """

    def __init__(
        self,
        chunks: Sequence[dict],
        k1: float = 1.5,
        b: float = 0.75,
    ):
        """Construye el índice léxico sobre los chunks del corpus.

        Args:
            chunks: Chunks con al menos ``text`` e identificadores.
            k1: Controla cuánto crece el aporte de repetir un término.
            b: Controla cuánto normaliza BM25 por longitud del documento.

        Returns:
            None. Deja el índice listo para usar con ``search``.
        """
        self.chunks = list(chunks)
        self.docs = [
            tokenize(chunk["text"])
            for chunk in self.chunks
        ]
        self.k1 = k1
        self.b = b

        # Usamos la librería real cuando está disponible; si no, mantenemos
        # el mismo contrato con una implementación local para smoke mode.
        try:
            from rank_bm25 import BM25Okapi

            self._model = BM25Okapi(
                self.docs,
                k1=k1,
                b=b,
            )
            self.backend_name = "rank-bm25-okapi"
            self._fallback = None
        except ImportError:
            self._model = None
            self.backend_name = "didactic-bm25-fallback"
            self._fallback = self._build_fallback()

    def _build_fallback(self) -> dict:
        """Precalcula estadísticas necesarias para el BM25 local.

        No recibe parámetros externos: usa los documentos ya guardados en
        ``self.docs`` durante la construcción del retriever.

        Returns:
            Diccionario con IDF por término y longitud media del corpus.

        Este fallback permite estudiar la mecánica sin depender de rank-bm25;
        no pretende reemplazar una implementación optimizada de producción.
        """
        document_frequency = Counter()

        # Document frequency cuenta en cuántos documentos aparece cada término.
        # Usamos set(doc) para no contar repeticiones dentro del mismo documento.
        for doc in self.docs:
            document_frequency.update(set(doc))

        document_count = max(1, len(self.docs))
        average_length = (
            sum(len(doc) for doc in self.docs)
            / document_count
        )

        idf = {
            term: math.log(
                1
                + (
                    document_count
                    - frequency
                    + 0.5
                )
                / (
                    frequency
                    + 0.5
                )
            )
            for term, frequency in document_frequency.items()
        }

        return {
            "idf": idf,
            "avgdl": average_length,
        }

    def _fallback_scores(
        self,
        query_tokens: list[str],
    ) -> list[float]:
        """Calcula un score BM25 por documento para una consulta tokenizada.

        Args:
            query_tokens: Tokens de la consulta normalizados con ``tokenize``.

        Returns:
            Lista de scores alineada por posición con ``self.docs``.
        """
        scores = []

        for doc in self.docs:
            frequencies = Counter(doc)
            score = 0.0

            # Cada término aporta según frecuencia local (TF), rareza global
            # (IDF) y una corrección por longitud del documento.
            for term in query_tokens:
                frequency = frequencies.get(term, 0)

                if not frequency:
                    continue

                denominator = (
                    frequency
                    + self.k1
                    * (
                        1
                        - self.b
                        + self.b
                        * len(doc)
                        / max(self._fallback["avgdl"], 1e-9)
                    )
                )

                score += (
                    self._fallback["idf"].get(term, 0.0)
                    * frequency
                    * (self.k1 + 1)
                    / denominator
                )

            scores.append(float(score))

        return scores

    def search(
        self,
        query: str,
        top_k: int = 5,
    ) -> list[LexicalResult]:
        """Recupera los chunks con mayor score BM25.

        Args:
            query: Consulta en lenguaje natural.
            top_k: Cantidad máxima de candidatos a devolver.

        Returns:
            ``LexicalResult`` ordenados de mayor a menor score.

        Esta salida todavía no se mezcla con Dense: la fusión ocurre en RRF.
        """
        query_tokens = tokenize(query)

        if self._model is not None:
            scores = self._model.get_scores(query_tokens)
        else:
            scores = self._fallback_scores(query_tokens)

        rows = [
            LexicalResult(
                chunk=chunk,
                score=float(score),
            )
            for chunk, score in zip(
                self.chunks,
                scores,
                strict=True,
            )
        ]

        return sorted(
            rows,
            key=lambda row: row.score,
            reverse=True,
        )[:top_k]



def _as_search_result(chunk: dict) -> SearchResult:
    """Adapta un chunk léxico al contrato común ``SearchResult``.

    Args:
        chunk: Chunk original recuperado por BM25.

    Returns:
        ``SearchResult`` con la misma identidad y metadata del chunk.

    RRF necesita un payload común para candidatos que pueden provenir de Dense,
    de BM25 o de ambos.
    """
    return SearchResult(
        chunk_id=chunk["chunk_id"],
        document_id=chunk["document_id"],
        text=chunk["text"],
        score=0.0,
        source=chunk["source"],
        page=chunk.get("page"),
        metadata=chunk.get("metadata", {}),
    )



def rrf_fuse(
    dense_results: Sequence[SearchResult],
    lexical_results: Sequence[LexicalResult],
    *,
    k: int = 60,
    top_k: int = 10,
) -> list[FusedCandidate]:
    """Fusiona Dense y BM25 con Reciprocal Rank Fusion (RRF).

    Args:
        dense_results: Ranking producido por búsqueda vectorial.
        lexical_results: Ranking producido por BM25.
        k: Constante de RRF que suaviza la ventaja de las primeras posiciones.
        top_k: Máximo de candidatos fusionados a devolver.

    Returns:
        Candidatos ordenados por ``rrf_score`` con el rank de cada señal.

    RRF usa posiciones y no scores crudos porque Dense y BM25 viven en escalas
    diferentes; sumarlos directamente no tendría una interpretación estable.
    """
    scores: dict[str, float] = {}
    payloads: dict[str, SearchResult] = {}
    dense_ranks: dict[str, int] = {}
    lexical_ranks: dict[str, int] = {}

    # Cada ranking aporta 1 / (k + rank). Un candidato presente en ambas
    # listas acumula evidencia de las dos señales.
    for rank, result in enumerate(dense_results, start=1):
        scores[result.chunk_id] = (
            scores.get(result.chunk_id, 0.0)
            + 1 / (k + rank)
        )
        payloads[result.chunk_id] = result
        dense_ranks[result.chunk_id] = rank

    for rank, lexical in enumerate(lexical_results, start=1):
        chunk = lexical.chunk
        chunk_id = chunk["chunk_id"]

        scores[chunk_id] = (
            scores.get(chunk_id, 0.0)
            + 1 / (k + rank)
        )
        payloads.setdefault(
            chunk_id,
            _as_search_result(chunk),
        )
        lexical_ranks[chunk_id] = rank

    # Ordenamos una sola vez después de acumular las contribuciones de ambos rankings.
    ordered = sorted(
        scores.items(),
        key=lambda item: item[1],
        reverse=True,
    )[:top_k]

    return [
        FusedCandidate(
            result=payloads[chunk_id],
            rrf_score=float(score),
            dense_rank=dense_ranks.get(chunk_id),
            lexical_rank=lexical_ranks.get(chunk_id),
        )
        for chunk_id, score in ordered
    ]


Overwriting ai_agent_project/src/ai_agent_course/ranking.py


In [37]:
# Recargamos el módulo recién escrito para que la notebook use su versión actual.
import ai_agent_course.ranking as ranking_module
importlib.reload(ranking_module)

BM25Retriever = ranking_module.BM25Retriever
rrf_fuse = ranking_module.rrf_fuse

# Construimos BM25 sobre exactamente los mismos chunks usados por Dense.
bm25 = BM25Retriever(CHUNKS)

print("BM25 backend:", bm25.backend_name)

sample_case = next(
    case
    for case in positive_cases
    if case["case_id"] == "EXP-01"
)

sample_query = sample_case["query"]

# Generamos dos rankings independientes y luego los fusionamos por posición.
dense_sample = dense_retriever.search(
    sample_query,
    top_k=6,
)
lexical_sample = bm25.search(
    sample_query,
    top_k=6,
)
fused_sample = rrf_fuse(
    dense_sample,
    lexical_sample,
    top_k=6,
)

print("CASO:", sample_case["case_id"])
print("QUERY:", sample_query)
print("GROUND TRUTH:", sample_case["relevant_document_ids"])

show_table([
    {
        "rank_rrf": rank,
        "document_id": candidate.result.document_id,
        "dense_rank": candidate.dense_rank,
        "lexical_rank": candidate.lexical_rank,
        "rrf_score": round(
            candidate.rrf_score,
            6,
        ),
    }
    for rank, candidate in enumerate(
        fused_sample,
        start=1,
    )
])


BM25 backend: rank-bm25-okapi
CASO: EXP-01
QUERY: ¿cuál es el plazo para presentar un reintegro?
GROUND TRUTH: ['POL-EXP-002']


,rank_rrf,document_id,dense_rank,lexical_rank,rrf_score
0,1,POL-BEN-007,1.0,3.0,0.032266
1,2,POL-EXP-002,3.0,2.0,0.032002
2,3,FAQ-RRHH-005,5.0,1.0,0.031778
3,4,POL-SEC-003,2.0,4.0,0.031754
4,5,POL-VAC-001,4.0,NaN,0.015625
5,6,INC-SEC-006,NaN,5.0,0.015385


## 3. Hybrid Search + RRF: recuperar amplio

**Hybrid Search** no significa “sumar BM25 + cosine similarity”.

Dense y BM25 producen scores con significado y escala diferentes. RRF evita normalizarlos porque usa únicamente el **rank**:

```text
RRF_score(d) = Σ 1 / (k + rank_i(d))
```

Un documento que aparece alto en ambas listas acumula evidencia de dos señales diferentes.

En esta fórmula, `k` es una **constante de suavizado del ranking**; no es el `top_k` de resultados. En el laboratorio usamos `k=60`, un valor habitual para que pequeñas diferencias de posición no dominen demasiado la fusión.

> El objetivo de esta etapa es construir un **candidate set** amplio y razonable.


In [38]:
# Comparamos las tres señales sobre un caso del ground truth heredado.
demo_case = next(
    case
    for case in positive_cases
    if case["case_id"] == "SEC-01"
)
query = demo_case["query"]

# Dense captura similitud semántica; BM25 captura coincidencia de términos.
# RRF combina ambos rankings sin exigir que sus scores sean comparables.
dense_rows = dense_retriever.search(
    query,
    top_k=6,
)
lexical_rows = bm25.search(
    query,
    top_k=6,
)
hybrid_rows = rrf_fuse(
    dense_rows,
    lexical_rows,
    top_k=6,
)

print("CASO:", demo_case["case_id"])
print("QUERY:", demo_case["query"])
print("GROUND TRUTH:", demo_case["relevant_document_ids"])

# Mostramos los rankings por separado para poder ver qué señal aportó cada candidato.
print("\nDense")
show_table([
    {
        "rank": rank,
        "document_id": row.document_id,
        "chunk_id": row.chunk_id,
        "score": round(float(row.score), 4),
    }
    for rank, row in enumerate(
        dense_rows,
        start=1,
    )
])

print("\nBM25")
show_table([
    {
        "rank": rank,
        "document_id": row.chunk["document_id"],
        "chunk_id": row.chunk["chunk_id"],
        "score": round(float(row.score), 4),
    }
    for rank, row in enumerate(
        lexical_rows,
        start=1,
    )
])

print("\nHybrid + RRF")
show_table([
    {
        "rank": rank,
        "document_id": row.result.document_id,
        "chunk_id": row.result.chunk_id,
        "dense_rank": row.dense_rank,
        "lexical_rank": row.lexical_rank,
        "rrf_score": round(
            row.rrf_score,
            6,
        ),
    }
    for rank, row in enumerate(
        hybrid_rows,
        start=1,
    )
])


CASO: SEC-01
QUERY: expuse mi contraseña, ¿qué acciones debo tomar?
GROUND TRUTH: ['POL-SEC-003', 'INC-SEC-006']

Dense


,rank,document_id,chunk_id,score
0,1,INC-SEC-006,INC-SEC-006:recursive:000:db0b5e7d,0.9231
1,2,POL-SEC-003,POL-SEC-003:recursive:000:8dd616f6,0.9172
2,3,POL-SEC-003,POL-SEC-003:recursive:001:b6ee93f8,0.8433
3,4,POL-EXP-002,POL-EXP-002:recursive:001:3ac2515d,0.0261
4,5,POL-BEN-007,POL-BEN-007:recursive:000:74c98ad3,0.0217
5,6,POL-REM-004,POL-REM-004:recursive:000:3ba7329a,0.0141



BM25


,rank,document_id,chunk_id,score
0,1,INC-SEC-006,INC-SEC-006:recursive:000:db0b5e7d,2.0837
1,2,POL-VAC-001,POL-VAC-001:recursive:000:f05eeb6e,0.0000
2,3,POL-VAC-001,POL-VAC-001:recursive:001:2bc935bb,0.0000
3,4,POL-EXP-002,POL-EXP-002:recursive:001:3ac2515d,0.0000
4,5,POL-SEC-003,POL-SEC-003:recursive:000:8dd616f6,0.0000
5,6,POL-SEC-003,POL-SEC-003:recursive:001:b6ee93f8,0.0000



Hybrid + RRF


,rank,document_id,chunk_id,dense_rank,lexical_rank,rrf_score
0,1,INC-SEC-006,INC-SEC-006:recursive:000:db0b5e7d,1.0,1,0.032787
1,2,POL-SEC-003,POL-SEC-003:recursive:000:8dd616f6,2.0,5,0.031514
2,3,POL-EXP-002,POL-EXP-002:recursive:001:3ac2515d,4.0,4,0.031250
3,4,POL-SEC-003,POL-SEC-003:recursive:001:b6ee93f8,3.0,6,0.031025
4,5,POL-VAC-001,POL-VAC-001:recursive:000:f05eeb6e,NaN,2,0.016129
5,6,POL-VAC-001,POL-VAC-001:recursive:001:2bc935bb,NaN,3,0.015873


### 3.1 Candidate set

El **candidate set** es el conjunto inicial de documentos recuperados **antes** del reranking.

Esta distinción es clave:

> El reranker puede cambiar el orden de los candidatos, pero **no puede recuperar un documento que nunca entró al candidate set**.

Por eso el diagnóstico correcto es:

- relevante **no aparece** → revisar retrieval, chunking, filtros o corpus;
- relevante **aparece pero queda abajo** → el reranking puede ayudar.


## 4. Re-ranking: recuperar amplio, leer fino

### Bi-Encoder vs. Cross-Encoder

En el dense retrieval que ya conocemos, el patrón habitual es un **Bi-Encoder**:

- la query se codifica por separado;
- cada chunk se codifica por separado;
- los embeddings de documentos se pueden precalcular;
- buscar entre muchos vectores es rápido.

Un **Cross-Encoder** hace otra cosa:

- recibe `query + candidato` juntos;
- calcula directamente qué tan relevante es ese par;
- suele ser más preciso para ordenar;
- es demasiado costoso para comparar contra todo el corpus.

Por eso usamos una arquitectura de dos etapas:

```text
Bi-Encoder / Hybrid → muchos candidatos → Cross-Encoder → pocos candidatos ordenados
```


### 4.1 Probar el contrato antes de descargar un modelo

En smoke mode usamos un reranker lexical muy simple para verificar:

- el formato del candidate set;
- la entrada `query + candidatos`;
- la salida ordenada;
- el `top_k`.

**No lo presentamos como equivalente semántico de un Cross-Encoder.**


In [39]:
from types import SimpleNamespace


class LexicalSmokeRerankerPreview:
    """Reranker mínimo para comprobar el contrato antes de cargar un modelo real.

    No intenta modelar relevancia semántica: solo permite verificar la idea de
    "recibir candidatos -> asignar score nuevo -> reordenar -> cortar top_k".
    """

    name = "lexical-overlap-smoke-only"

    def rerank(self, query: str, candidates: list, top_k: int = 3):
        """Reordena candidatos por solapamiento léxico.

        Args:
            query: Consulta original.
            candidates: Candidate set producido por la etapa anterior.
            top_k: Cantidad de elementos que conserva el reranker.

        Returns:
            Filas ordenadas por un score didáctico de palabras compartidas.
        """
        query_tokens = set(re.findall(r"\w+", query.lower()))
        rows = []

        # A diferencia de retrieval, el reranker solo trabaja sobre un conjunto
        # pequeño de candidatos ya recuperados: no recorre todo el corpus.
        for candidate in candidates:
            candidate_tokens = set(
                re.findall(r"\w+", candidate.result.text.lower())
            )
            overlap = len(query_tokens & candidate_tokens)

            rows.append({
                "candidate": candidate,
                "rerank_score": float(overlap),
            })

        return sorted(
            rows,
            key=lambda row: row["rerank_score"],
            reverse=True,
        )[:top_k]


preview_candidates = [
    SimpleNamespace(
        result=SimpleNamespace(
            text="política general de personas",
            chunk_id="general",
        ),
        rrf_score=0.90,
    ),
    SimpleNamespace(
        result=SimpleNamespace(
            text="vacaciones licencia anual anticipación",
            chunk_id="vac",
        ),
        rrf_score=0.70,
    ),
]

preview_reranker = LexicalSmokeRerankerPreview()
reranked_preview = preview_reranker.rerank(
    "anticipación vacaciones",
    preview_candidates,
)

assert reranked_preview[0]["candidate"].result.chunk_id == "vac"
reranked_preview


[{'candidate': namespace(result=namespace(text='vacaciones licencia anual anticipación',
                             chunk_id='vac'),
            rrf_score=0.7),
  'rerank_score': 2.0},
 {'candidate': namespace(result=namespace(text='política general de personas',
                             chunk_id='general'),
            rrf_score=0.9),
  'rerank_score': 0.0}]

### 4.2 Persistir el reranker

El módulo mantiene dos implementaciones con el mismo contrato:

- `CrossEncoderReranker`: modelo real;
- `LexicalSmokeReranker`: fallback explícito para ejecución offline.

La notebook muestra siempre cuál quedó activo.


In [40]:
%%writefile ai_agent_project/src/ai_agent_course/reranker.py
from __future__ import annotations

from dataclasses import dataclass
import re
from typing import Sequence

from .ranking import FusedCandidate
from .retrieval import SearchResult


@dataclass(frozen=True)
class RankedCandidate:
    """Resultado reordenado que conserva el score RRF para trazabilidad."""

    result: SearchResult
    rerank_score: float
    rrf_score: float


class CrossEncoderReranker:
    """Reranker que puntúa ``query + texto`` conjuntamente con un Cross-Encoder.

    A diferencia de embeddings bi-encoder, el modelo observa ambos textos a la
    vez. Eso suele mejorar el orden fino, pero agrega costo por cada candidato.
    """

    default_model = "cross-encoder/ms-marco-MiniLM-L-6-v2"

    def __init__(
        self,
        model_name: str = default_model,
    ):
        """Carga el Cross-Encoder que se usará para reordenar candidatos.

        Args:
            model_name: Identificador del modelo compatible con sentence-transformers.

        Returns:
            None. Deja el modelo disponible en ``self.model``.
        """
        from sentence_transformers import CrossEncoder

        self.model = CrossEncoder(model_name)
        self.name = model_name

    def rerank(
        self,
        query: str,
        candidates: Sequence[FusedCandidate],
        top_k: int = 3,
    ) -> list[RankedCandidate]:
        """Reordena un candidate set con un score de relevancia query-documento.

        Args:
            query: Consulta original del usuario.
            candidates: Candidatos ya recuperados y fusionados por RRF.
            top_k: Cantidad final de resultados que se conservará.

        Returns:
            ``RankedCandidate`` ordenados por score del Cross-Encoder.
        """
        # Cada inferencia compara la consulta con un candidato. Por eso conviene
        # rerankear pocos elementos y no usar este modelo sobre todo el corpus.
        pairs = [
            (query, candidate.result.text)
            for candidate in candidates
        ]

        scores = self.model.predict(pairs)

        rows = [
            RankedCandidate(
                result=candidate.result,
                rerank_score=float(score),
                rrf_score=float(candidate.rrf_score),
            )
            for candidate, score in zip(
                candidates,
                scores,
                strict=True,
            )
        ]

        return sorted(
            rows,
            key=lambda row: row.rerank_score,
            reverse=True,
        )[:top_k]


class LexicalSmokeReranker:
    """Fallback offline que conserva el contrato sin simular semántica real."""

    name = "lexical-overlap-smoke-only"

    def rerank(
        self,
        query: str,
        candidates: Sequence[FusedCandidate],
        top_k: int = 3,
    ) -> list[RankedCandidate]:
        """Reordena candidatos con una señal léxica simple para smoke mode.

        Args:
            query: Consulta original.
            candidates: Candidate set producido por RRF.
            top_k: Cantidad final de elementos a devolver.

        Returns:
            ``RankedCandidate`` compatibles con la salida del Cross-Encoder.
        """
        query_tokens = set(
            re.findall(r"\w+", query.lower())
        )

        rows = []

        for candidate in candidates:
            candidate_tokens = set(
                re.findall(
                    r"\w+",
                    candidate.result.text.lower(),
                )
            )

            overlap = len(
                query_tokens & candidate_tokens
            )

            rows.append(
                RankedCandidate(
                    result=candidate.result,
                    rerank_score=float(overlap),
                    rrf_score=float(candidate.rrf_score),
                )
            )

        return sorted(
            rows,
            key=lambda row: row.rerank_score,
            reverse=True,
        )[:top_k]


Overwriting ai_agent_project/src/ai_agent_course/reranker.py


In [41]:
# Seleccionamos el reranker sin romper la reproducibilidad del laboratorio.
# Smoke mode siempre usa el fallback; fuera de él intentamos cargar el Cross-Encoder real.
import ai_agent_course.reranker as reranker_module
importlib.reload(reranker_module)

reranker = reranker_module.LexicalSmokeReranker()

if not SMOKE_MODE:
    if not HAS_SENTENCE_TRANSFORMERS:
        print(
            "sentence-transformers no está instalado; "
            "se mantiene el fallback explícito."
        )
    else:
        try:
            reranker = reranker_module.CrossEncoderReranker(
                model_name=os.getenv(
                    "CROSS_ENCODER_MODEL",
                    reranker_module.CrossEncoderReranker.default_model,
                )
            )
        except (RuntimeError, OSError, ValueError) as exc:
            print(
                "Cross-Encoder no disponible; "
                "se mantiene fallback explícito:",
                type(exc).__name__,
            )

# El reranker recibe el candidate set híbrido y produce el ranking final.
reranked_sample = reranker.rerank(
    query,
    hybrid_rows,
    top_k=3,
)

print("Reranker activo:", reranker.name)

show_table([
    {
        "rank": rank,
        "document_id": row.result.document_id,
        "chunk_id": row.result.chunk_id,
        "rerank_score": round(float(row.rerank_score), 4),
        "rrf_score": round(float(row.rrf_score), 6),
    }
    for rank, row in enumerate(reranked_sample, start=1)
])


/Users/gasparcartasso/Desktop/AI Developer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 11553.65it/s]


Reranker activo: cross-encoder/ms-marco-MiniLM-L-6-v2


,rank,document_id,chunk_id,rerank_score,rrf_score
0,1,INC-SEC-006,INC-SEC-006:recursive:000:db0b5e7d,-4.1156,0.032787
1,2,POL-SEC-003,POL-SEC-003:recursive:001:b6ee93f8,-9.3110,0.031025
2,3,POL-EXP-002,POL-EXP-002:recursive:001:3ac2515d,-9.8335,0.031250


## 5. Una fachada de búsqueda, tres estrategias internas

Ahora encapsulamos las tres variantes detrás del mismo contrato:

```text
search_course(query) → evidencia
```

La aplicación consumidora no debería saber si internamente usamos:

- dense;
- dense + BM25 + RRF;
- hybrid + reranking.

Esta separación permite mejorar la confiabilidad **sin romper a quien consume la interfaz**.


In [42]:
%%writefile ai_agent_project/src/ai_agent_course/course_search.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Literal

from .ranking import BM25Retriever, rrf_fuse


SearchStrategy = Literal[
    "dense",
    "hybrid_rrf",
    "hybrid_rerank",
]


@dataclass
class SearchTrace:
    """Traza de diagnóstico que registra cuántos candidatos pasó cada etapa.

    No cambia el ranking: existe para observabilidad y para explicar qué camino
    siguió una búsqueda sin acoplar esa información al resultado final.
    """

    strategy: SearchStrategy
    dense_candidates: int
    lexical_candidates: int
    fused_candidates: int
    reranked_candidates: int
    reranker: str | None


class CourseSearchService:
    """Orquesta Dense, BM25, RRF y reranking detrás de una interfaz estable.

    La aplicación llama siempre a ``search``. La estrategia puede cambiar sin
    obligar al consumidor a conocer cómo se implementa cada etapa interna.
    """

    def __init__(
        self,
        dense_retriever,
        chunks,
        reranker,
        *,
        strategy: SearchStrategy = "hybrid_rerank",
        candidate_k: int = 8,
    ):
        """Configura los componentes internos del servicio de búsqueda.

        Args:
            dense_retriever: Retriever vectorial ya indexado.
            chunks: Corpus usado para construir el retriever BM25.
            reranker: Componente con método ``rerank`` y atributo ``name``.
            strategy: Estrategia usada por defecto.
            candidate_k: Tamaño del conjunto intermedio antes del top-k final.

        Returns:
            None. Deja listo el servicio para responder ``search``.
        """
        self.dense_retriever = dense_retriever
        self.lexical_retriever = BM25Retriever(chunks)
        self.reranker = reranker
        self.strategy = strategy
        self.candidate_k = candidate_k

    def search(
        self,
        query: str,
        top_k: int = 3,
        *,
        strategy: SearchStrategy | None = None,
    ):
        """Ejecuta la estrategia elegida manteniendo un contrato de salida común.

        Args:
            query: Consulta de búsqueda.
            top_k: Cantidad final de resultados.
            strategy: Override opcional de la estrategia configurada.

        Returns:
            Tupla ``(results, trace)``. ``results`` contiene SearchResult y
            ``trace`` describe cuántos candidatos recorrieron cada etapa.
        """
        chosen = strategy or self.strategy

        # Dense se ejecuta siempre: es el baseline y también una de las dos
        # señales que alimentan las variantes híbridas.
        dense = self.dense_retriever.search(
            query,
            top_k=max(self.candidate_k, top_k),
        )

        if chosen == "dense":
            return dense[:top_k], SearchTrace(
                strategy=chosen,
                dense_candidates=len(dense),
                lexical_candidates=0,
                fused_candidates=0,
                reranked_candidates=0,
                reranker=None,
            )

        # Solo las estrategias híbridas agregan la señal léxica de BM25.
        lexical = self.lexical_retriever.search(
            query,
            top_k=max(self.candidate_k, top_k),
        )

        # RRF construye un candidate set común antes de decidir si termina
        # aquí o si pasa por una etapa adicional de reranking.
        fused = rrf_fuse(
            dense,
            lexical,
            top_k=max(self.candidate_k, top_k),
        )

        if chosen == "hybrid_rrf":
            return (
                [candidate.result for candidate in fused[:top_k]],
                SearchTrace(
                    strategy=chosen,
                    dense_candidates=len(dense),
                    lexical_candidates=len(lexical),
                    fused_candidates=len(fused),
                    reranked_candidates=0,
                    reranker=None,
                ),
            )

        if chosen != "hybrid_rerank":
            raise ValueError(f"Estrategia desconocida: {chosen}")

        # El reranker opera sobre pocos candidatos para mejorar el orden fino
        # sin pagar el costo de un Cross-Encoder sobre todo el corpus.
        reranked = self.reranker.rerank(
            query,
            fused,
            top_k=top_k,
        )

        return (
            [candidate.result for candidate in reranked],
            SearchTrace(
                strategy=chosen,
                dense_candidates=len(dense),
                lexical_candidates=len(lexical),
                fused_candidates=len(fused),
                reranked_candidates=len(reranked),
                reranker=self.reranker.name,
            ),
        )


Overwriting ai_agent_project/src/ai_agent_course/course_search.py


In [43]:
# Instanciamos una fachada única de búsqueda para desacoplar a la aplicación
# de la estrategia interna que estamos experimentando.
import ai_agent_course.course_search as course_search_module
importlib.reload(course_search_module)

search_service = course_search_module.CourseSearchService(
    dense_retriever=dense_retriever,
    chunks=CHUNKS,
    reranker=reranker,
    strategy="hybrid_rerank",
    candidate_k=8,
)

def _serialize_results(results) -> list[dict]:
    """Convierte resultados internos a una estructura serializable y estable.

    Args:
        results: Secuencia de objetos ``SearchResult``.

    Returns:
        Lista de diccionarios con ids, fuente, texto y metadata.

    Esta frontera evita exponer dataclasses internas a quien consuma la capacidad.
    """
    return [
        {
            "document_id": row.document_id,
            "chunk_id": row.chunk_id,
            "source": row.source,
            "page": row.page,
            "text": row.text,
            "metadata": row.metadata,
        }
        for row in results
    ]

def search_course(
    query: str,
    top_k: int = 3,
) -> list[dict]:
    """Busca evidencia interna detrás de una interfaz estable.

    Args:
        query: Consulta del usuario.
        top_k: Cantidad máxima de evidencias a devolver.

    Returns:
        Lista serializable de resultados con texto, ids, fuente y metadata.

    La aplicación consumidora no necesita saber si la implementación usa
    Dense, BM25, RRF o Cross-Encoder; esa decisión queda encapsulada.
    """
    # La traza queda disponible para diagnóstico, pero esta interfaz pública
    # devuelve solo evidencia para mantener un contrato simple.
    results, _trace = search_service.search(
        query,
        top_k=top_k,
    )
    return _serialize_results(results)

search_course(
    baseline_case["query"],
    top_k=3,
)


[{'document_id': 'FAQ-RRHH-005',
  'chunk_id': 'FAQ-RRHH-005:recursive:000:8127fc62',
  'source': 'faq_rrhh.md',
  'page': 1,
  'text': '# Preguntas frecuentes de RRHH Para pedir vacaciones, registrá la solicitud con 15 días corridos de anticipación. La carga no equivale a aprobación: el líder revisa cobertura y confirma la decisión. Para modificar una solicitud aprobada, cancelá la fecha original y generá una nueva solicitud.',
  'metadata': {'topic': 'vacaciones',
   'department': 'RRHH',
   'active': True,
   'version': '2026.1',
   'chunk_strategy': 'recursive',
   'chunk_index': 0}},
 {'document_id': 'POL-VAC-001',
  'chunk_id': 'POL-VAC-001:recursive:000:f05eeb6e',
  'source': 'politica_vacaciones.md',
  'page': 1,
  'text': 'Las solicitudes de vacaciones deben registrarse en la herramienta de RRHH con al menos 15 días corridos de anticipación. La carga inicia el circuito de revisión, pero no implica aprobación automática. El líder directo valida la cobertura operativa y las fech

## 6. Evaluar y diagnosticar las estrategias

Con las tres estrategias detrás del mismo contrato, ahora las comparamos bajo condiciones controladas.

La pregunta es:

> Si mantengo corpus, queries, ground truth y embeddings, ¿qué cambia cuando modifico solamente la estrategia de retrieval/ranking?

Comparamos:

- **A. Dense baseline**
- **B. Hybrid + RRF**
- **C. Hybrid + reranking**

Y conservamos las métricas ya conocidas:

- **Precision@k** → cuánto ruido entra;
- **Recall@k** → cuánta evidencia esperada recuperamos;
- **Hit Rate@k** → si apareció al menos un relevante;
- **MRR** → qué tan arriba aparece el primer relevante;
- **Latencia** → costo operacional de la mejora.

Medimos `k = 1, 3, 5`.

El caso `NONE-01` se analiza aparte porque su ground truth es justamente **no tener evidencia**.


In [44]:
# Reutilizamos la lógica de evaluación de C6 a nivel DOCUMENTO.
def unique_document_ids(results, k: int) -> list[str]:
    """Colapsa chunks al nivel documento sin perder el orden del ranking.

    Args:
        results: Resultados rankeados a nivel chunk.
        k: Máximo de documentos únicos a conservar.

    Returns:
        Lista de ``document_id`` únicos en el orden en que aparecen.

    Evaluamos a nivel documento porque el ground truth de C6 está definido así.
    """
    ids = []

    for result in results:
        if result.document_id not in ids:
            ids.append(result.document_id)

        if len(ids) == k:
            break

    return ids


def precision_at_k(
    results,
    relevant_ids,
    k: int,
) -> float:
    """Calcula Precision@k a nivel documento.

    Args:
        results: Ranking de resultados.
        relevant_ids: Documentos considerados relevantes por el golden.
        k: Corte del ranking.

    Returns:
        Fracción de documentos recuperados que son relevantes.
    """
    retrieved = unique_document_ids(results, k)

    if not retrieved:
        return 0.0

    relevant = set(relevant_ids)
    return (
        sum(doc_id in relevant for doc_id in retrieved)
        / len(retrieved)
    )


def recall_at_k(
    results,
    relevant_ids,
    k: int,
) -> float:
    """Calcula Recall@k a nivel documento.

    Args:
        results: Ranking de resultados.
        relevant_ids: Documentos relevantes esperados.
        k: Corte del ranking.

    Returns:
        Fracción del ground truth presente en los primeros k documentos.
    """
    relevant = set(relevant_ids)

    if not relevant:
        return 0.0

    retrieved = set(
        unique_document_ids(results, k)
    )
    return len(retrieved & relevant) / len(relevant)


def hit_rate_at_k(
    results,
    relevant_ids,
    k: int,
) -> float:
    """Calcula Hit Rate@k para un caso individual.

    Args:
        results: Ranking de resultados.
        relevant_ids: Documentos relevantes esperados.
        k: Corte del ranking.

    Returns:
        1.0 si aparece al menos un documento relevante en top-k; 0.0 si no.
    """
    relevant = set(relevant_ids)
    retrieved = set(
        unique_document_ids(results, k)
    )
    return float(bool(retrieved & relevant))


def reciprocal_rank(
    results,
    relevant_ids,
) -> float:
    """Calcula Reciprocal Rank del primer documento relevante.

    Args:
        results: Ranking a nivel chunk.
        relevant_ids: Documentos relevantes del caso.

    Returns:
        ``1 / rank`` del primer documento relevante, o 0.0 si no aparece.

    ``logical_rank`` ignora chunks repetidos del mismo documento para mantener
    coherencia con la evaluación documental usada en el resto de la notebook.
    """
    relevant = set(relevant_ids)
    seen = set()
    logical_rank = 0

    for result in results:
        if result.document_id in seen:
            continue

        seen.add(result.document_id)
        logical_rank += 1

        if result.document_id in relevant:
            return 1.0 / logical_rank

    return 0.0


STRATEGIES = [
    "dense",
    "hybrid_rrf",
    "hybrid_rerank",
]

K_VALUES = (1, 3, 5)
MAX_K = max(K_VALUES)

# Guardamos los rankings para no repetir búsquedas al cambiar k.
strategy_results = {
    strategy: {}
    for strategy in STRATEGIES
}
comparison_rows = []
latency_rows = []

# Ejecutamos cada estrategia una sola vez por caso con el mayor k necesario.
# Luego derivamos @1, @3 y @5 del mismo ranking para comparar de forma consistente.
for case in positive_cases:
    for strategy in STRATEGIES:
        # Medimos latencia end-to-end de la estrategia, incluyendo sus etapas internas.
        started = time.perf_counter()

        results, trace = search_service.search(
            case["query"],
            top_k=MAX_K,
            strategy=strategy,
        )

        latency_ms = (
            time.perf_counter() - started
        ) * 1000

        strategy_results[strategy][
            case["case_id"]
        ] = results

        latency_rows.append({
            "case_id": case["case_id"],
            "strategy": strategy,
            "latency_ms": round(
                latency_ms,
                3,
            ),
        })

        row = {
            "case_id": case["case_id"],
            "category": case["category"],
            "strategy": strategy,
            "top_docs": unique_document_ids(
                results,
                MAX_K,
            ),
            "mrr": round(
                reciprocal_rank(
                    results,
                    case["relevant_document_ids"],
                ),
                3,
            ),
            "latency_ms": round(
                latency_ms,
                3,
            ),
            "trace": trace.__dict__,
        }

        # Calculamos varias métricas sobre el mismo resultado: cobertura, precisión
        # y presencia de al menos un relevante responden preguntas diferentes.
        for k in K_VALUES:
            row[f"precision@{k}"] = round(
                precision_at_k(
                    results,
                    case["relevant_document_ids"],
                    k,
                ),
                3,
            )
            row[f"recall@{k}"] = round(
                recall_at_k(
                    results,
                    case["relevant_document_ids"],
                    k,
                ),
                3,
            )
            row[f"hit@{k}"] = hit_rate_at_k(
                results,
                case["relevant_document_ids"],
                k,
            )

        comparison_rows.append(row)

show_table(
    comparison_rows,
    [
        "case_id",
        "category",
        "strategy",
        "precision@1",
        "recall@1",
        "hit@1",
        "precision@3",
        "recall@3",
        "hit@3",
        "precision@5",
        "recall@5",
        "hit@5",
        "mrr",
        "latency_ms",
    ],
)


,case_id,category,strategy,precision@1,recall@1,hit@1,precision@3,recall@3,hit@3,precision@5,recall@5,hit@5,mrr,latency_ms
0,VAC-01,vacaciones,dense,1.0,0.5,1.0,0.667,1.0,1.0,0.5,1.0,1.0,1.000,0.171
1,VAC-01,vacaciones,hybrid_rrf,1.0,0.5,1.0,0.667,1.0,1.0,0.5,1.0,1.0,1.000,0.174
2,VAC-01,vacaciones,hybrid_rerank,1.0,0.5,1.0,0.667,1.0,1.0,0.4,1.0,1.0,1.000,20.536
3,EXP-01,gastos,dense,0.0,0.0,0.0,0.333,1.0,1.0,0.2,1.0,1.0,0.333,0.301
4,EXP-01,gastos,hybrid_rrf,0.0,0.0,0.0,0.333,1.0,1.0,0.2,1.0,1.0,0.500,0.359
5,EXP-01,gastos,hybrid_rerank,0.0,0.0,0.0,0.333,1.0,1.0,0.2,1.0,1.0,0.500,148.803
6,SEC-01,seguridad,dense,1.0,0.5,1.0,0.667,1.0,1.0,0.5,1.0,1.0,1.000,0.142
7,SEC-01,seguridad,hybrid_rrf,1.0,0.5,1.0,0.667,1.0,1.0,0.5,1.0,1.0,1.000,0.146
8,SEC-01,seguridad,hybrid_rerank,1.0,0.5,1.0,0.667,1.0,1.0,0.5,1.0,1.0,1.000,26.333
9,REM-01,trabajo remoto,dense,1.0,1.0,1.0,0.333,1.0,1.0,0.2,1.0,1.0,1.000,0.127


In [45]:
# Agregamos por estrategia y por k, igual que en C6 pero comparando variantes.
aggregate_rows = []

# Primero resumimos Precision, Recall y Hit Rate para cada corte k.
for strategy in STRATEGIES:
    rows = [
        row
        for row in comparison_rows
        if row["strategy"] == strategy
    ]

    for k in K_VALUES:
        aggregate_rows.append({
            "strategy": strategy,
            "k": k,
            "mean_precision": round(
                statistics.mean(
                    row[f"precision@{k}"]
                    for row in rows
                ),
                3,
            ),
            "mean_recall": round(
                statistics.mean(
                    row[f"recall@{k}"]
                    for row in rows
                ),
                3,
            ),
            "hit_rate": round(
                statistics.mean(
                    row[f"hit@{k}"]
                    for row in rows
                ),
                3,
            ),
        })

mrr_latency_summary = []

# MRR resume el orden del primer relevante; la latencia cuantifica el costo extra.
for strategy in STRATEGIES:
    rows = [
        row
        for row in comparison_rows
        if row["strategy"] == strategy
    ]

    latencies = [
        row["latency_ms"]
        for row in rows
    ]

    mrr_latency_summary.append({
        "strategy": strategy,
        "mean_mrr": round(
            statistics.mean(
                row["mrr"]
                for row in rows
            ),
            3,
        ),
        "mean_latency_ms": round(
            statistics.mean(latencies),
            3,
        ),
        "median_latency_ms": round(
            statistics.median(latencies),
            3,
        ),
        "max_latency_ms": round(
            max(latencies),
            3,
        ),
    })

print("Precision / Recall / Hit Rate por k")
show_table(aggregate_rows)

print("MRR y latencia")
show_table(mrr_latency_summary)


Precision / Recall / Hit Rate por k


,strategy,k,mean_precision,mean_recall,hit_rate
0,dense,1,0.800,0.6,0.8
1,dense,3,0.467,1.0,1.0
2,dense,5,0.320,1.0,1.0
3,hybrid_rrf,1,0.800,0.6,0.8
4,hybrid_rrf,3,0.467,1.0,1.0
5,hybrid_rrf,5,0.320,1.0,1.0
6,hybrid_rerank,1,0.600,0.4,0.6
7,hybrid_rerank,3,0.467,1.0,1.0
8,hybrid_rerank,5,0.300,1.0,1.0


MRR y latencia


,strategy,mean_mrr,mean_latency_ms,median_latency_ms,max_latency_ms
0,dense,0.867,0.181,0.163,0.301
1,hybrid_rrf,0.900,0.194,0.153,0.359
2,hybrid_rerank,0.800,73.251,26.333,148.866


### 6.1 Elegir técnica por síntoma, no por sofisticación

Ahora sí podemos diagnosticar con evidencia.

| Síntoma | Qué mirar | Intervención probable |
|---|---|---|
| `Recall@5` bajo | el relevante nunca entra | corpus, chunking, filtros, embeddings |
| `Recall@5` alto pero `Recall@1` bajo | entra, pero demasiado abajo | reranking |
| Precision baja | entra mucho ruido | ranking, filtros, chunking |
| códigos / siglas / nombres exactos fallan | Dense vs BM25 | sumar señal lexical |
| paráfrasis funcionan bien | desempeño Dense | mantener señal semántica |
| MRR mejora con reranker | primer relevante sube | reranking justificado |
| no existe evidencia | control negativo | abstención / otra capacidad |

**Regla:** no se acepta “Hybrid + Rerank porque es mejor”.  
Cada componente debe responder a una falla observada en el benchmark.


In [46]:
# Diagnóstico por caso y estrategia.
diagnostic_rows = []

# El diagnóstico traduce métricas a síntomas de arquitectura.
# No elige un ganador universal: sugiere qué técnica vale la pena investigar.
for case in positive_cases:
    case_rows = [
        row
        for row in comparison_rows
        if row["case_id"] == case["case_id"]
    ]

    dense_row = next(
        row
        for row in case_rows
        if row["strategy"] == "dense"
    )
    hybrid_row = next(
        row
        for row in case_rows
        if row["strategy"] == "hybrid_rrf"
    )
    rerank_row = next(
        row
        for row in case_rows
        if row["strategy"] == "hybrid_rerank"
    )

    # Orden de lectura:
    # 1) sin cobertura -> ampliar candidate set/señales;
    # 2) con cobertura pero mal rank -> reranking;
    # 3) con ruido -> revisar precisión/filtros;
    # 4) baseline suficiente -> exigir justificación al costo adicional.
    if dense_row["recall@5"] == 0:
        diagnosis = (
            "Dense no encuentra evidencia: revisar cobertura "
            "y observar si la señal lexical rescata candidatos."
        )
    elif dense_row["recall@1"] < dense_row["recall@5"]:
        diagnosis = (
            "La evidencia existe pero rankea abajo: "
            "reranking es candidato natural."
        )
    elif dense_row["precision@3"] < 0.5:
        diagnosis = (
            "Hay evidencia pero también ruido: "
            "comparar ranking y filtros."
        )
    else:
        diagnosis = (
            "Dense ya es competitivo; una técnica más compleja "
            "debe justificar su costo."
        )

    diagnostic_rows.append({
        "case_id": case["case_id"],
        "category": case["category"],
        "dense_recall@1": dense_row["recall@1"],
        "dense_recall@5": dense_row["recall@5"],
        "hybrid_recall@5": hybrid_row["recall@5"],
        "rerank_mrr": rerank_row["mrr"],
        "diagnosis": diagnosis,
    })

show_table(diagnostic_rows)

# El control negativo se evalúa aparte: un retriever siempre puede devolver
# "lo más parecido", aunque el corpus no contenga una respuesta válida.
# Este caso recuerda que retrieval necesita abstención/validación aguas abajo.
negative_case = no_evidence_cases[0]
negative_rows = []

for strategy in STRATEGIES:
    results, trace = search_service.search(
        negative_case["query"],
        top_k=5,
        strategy=strategy,
    )

    negative_rows.append({
        "strategy": strategy,
        "query": negative_case["query"],
        "top_docs": unique_document_ids(
            results,
            5,
        ),
        "returned_candidates": len(results),
        "note": (
            "El ranking devuelve lo más parecido, "
            "pero el ground truth indica que no existe evidencia."
        ),
    })

print("\nControl negativo — NONE-01")
show_table(negative_rows)


,case_id,category,dense_recall@1,dense_recall@5,hybrid_recall@5,rerank_mrr,diagnosis
0,VAC-01,vacaciones,0.5,1.0,1.0,1.0,La evidencia existe pero rankea abajo: reranki...
1,EXP-01,gastos,0.0,1.0,1.0,0.5,La evidencia existe pero rankea abajo: reranki...
2,SEC-01,seguridad,0.5,1.0,1.0,1.0,La evidencia existe pero rankea abajo: reranki...
3,REM-01,trabajo remoto,1.0,1.0,1.0,1.0,Hay evidencia pero también ruido: comparar ran...
4,CAP-01,capacitación,1.0,1.0,1.0,0.5,Hay evidencia pero también ruido: comparar ran...



Control negativo — NONE-01


,strategy,query,top_docs,returned_candidates,note
0,dense,¿la empresa ofrece estacionamiento gratuito?,"[FAQ-RRHH-005, POL-BEN-007, POL-VAC-001, POL-S...",5,"El ranking devuelve lo más parecido, pero el g..."
1,hybrid_rrf,¿la empresa ofrece estacionamiento gratuito?,"[FAQ-RRHH-005, POL-BEN-007, POL-VAC-001, POL-S...",5,"El ranking devuelve lo más parecido, pero el g..."
2,hybrid_rerank,¿la empresa ofrece estacionamiento gratuito?,"[POL-BEN-007, POL-REM-004, POL-VAC-001, INC-SE...",5,"El ranking devuelve lo más parecido, pero el g..."


## 7. Orden del contexto — intuición previa a Context Precision

Con el benchmark anterior ya distinguimos **cobertura** de **orden**.

El reranking no sirve solamente para que un documento relevante aparezca: también busca colocar evidencia útil **antes que el ruido**.

Acá hacemos una inspección didáctica, no una implementación formal de la métrica Context Precision:

- marcamos cada posición del ranking como `relevante / no relevante`;
- observamos la precisión acumulada en los primeros puestos;
- comparamos el orden antes y después del reranking.

Esto prepara la intuición de **Context Precision** sin confundirla con su medición formal, que queda para el bloque de Producción/LLMOps junto con Faithfulness, jueces y observabilidad.


In [47]:
def relevance_trace(results, relevant_document_ids, top_k: int = 5):
    """Construye una traza simple de relevancia posición por posición.

    Args:
        results: Ranking producido por una estrategia.
        relevant_document_ids: Ground truth documental del caso.
        top_k: Cantidad de posiciones a inspeccionar.

    Returns:
        Filas con rank, relevancia y precisión acumulada.

    No reemplaza una métrica formal de Context Precision; sirve para visualizar
    por qué mover evidencia útil hacia arriba mejora el contexto que verá el LLM.
    """
    relevant = set(relevant_document_ids)
    rows = []
    hits = 0

    for rank, document_id in enumerate(
        unique_document_ids(results, top_k),
        start=1,
    ):
        is_relevant = document_id in relevant
        hits += int(is_relevant)

        rows.append({
            "rank": rank,
            "document_id": document_id,
            "relevant": is_relevant,
            "precision_so_far": round(hits / rank, 3),
        })

    return rows


# Usamos el mismo caso para comparar orden antes y después del reranking.
focus_case = positive_cases[0]

hybrid_focus, _ = search_service.search(
    focus_case["query"],
    top_k=5,
    strategy="hybrid_rrf",
)

rerank_focus, _ = search_service.search(
    focus_case["query"],
    top_k=5,
    strategy="hybrid_rerank",
)

print("Hybrid + RRF")
show_table(
    relevance_trace(
        hybrid_focus,
        focus_case["relevant_document_ids"],
        top_k=5,
    )
)

print("Hybrid + reranking")
show_table(
    relevance_trace(
        rerank_focus,
        focus_case["relevant_document_ids"],
        top_k=5,
    )
)


Hybrid + RRF


,rank,document_id,relevant,precision_so_far
0,1,FAQ-RRHH-005,True,1.000
1,2,POL-VAC-001,True,1.000
2,3,POL-EXP-002,False,0.667
3,4,POL-BEN-007,False,0.500


Hybrid + reranking


,rank,document_id,relevant,precision_so_far
0,1,FAQ-RRHH-005,True,1.000
1,2,POL-VAC-001,True,1.000
2,3,POL-SEC-003,False,0.667
3,4,POL-BEN-007,False,0.500
4,5,POL-EXP-002,False,0.400


## 8. RAG Tournament — cierre del bloque técnico

Antes de introducir routing, cerramos el bloque de retrieval con una decisión de arquitectura basada en evidencia.

Cada grupo recibe uno o más casos del benchmark y compara:

- Dense;
- Hybrid + RRF;
- Hybrid + Rerank.

La decisión debe basarse en:

- Precision@k;
- Recall@k;
- Hit Rate@k;
- MRR;
- latencia;
- inspección del ranking.

### Preguntas obligatorias

1. ¿El relevante entra al candidate set?
2. ¿Aparece demasiado abajo?
3. ¿BM25 agrega una señal útil?
4. ¿El reranker mejora realmente el orden?
5. ¿El costo extra está justificado?
6. ¿Qué ocurre con el caso sin evidencia?

No se acepta:

> “Hybrid + Rerank porque es mejor.”

El objetivo es elegir una arquitectura **por síntoma medido**.


In [48]:
# Canvas editable para la actividad usando el ground truth heredado.
# Prellenamos únicamente datos observables y ground truth.
# Las columnas vacías obligan al grupo a registrar síntoma, evidencia y trade-off.
TOURNAMENT_CANVAS = [
    {
        "case_id": case["case_id"],
        "category": case["category"],
        "query": case["query"],
        "expects_evidence": case["expects_evidence"],
        "ground_truth": case["relevant_document_ids"],
        "observed_failure": "",
        "chosen_strategy": "",
        "metric_evidence": "",
        "cost_tradeoff": "",
        "decision": "",
    }
    for case in RETRIEVAL_GOLDEN
]

show_table(
    TOURNAMENT_CANVAS,
    [
        "case_id",
        "category",
        "expects_evidence",
        "ground_truth",
        "observed_failure",
        "chosen_strategy",
        "metric_evidence",
        "cost_tradeoff",
        "decision",
    ],
)


,case_id,category,expects_evidence,ground_truth,observed_failure,chosen_strategy,metric_evidence,cost_tradeoff,decision
0,VAC-01,vacaciones,True,"[POL-VAC-001, FAQ-RRHH-005]",,,,,
1,EXP-01,gastos,True,[POL-EXP-002],,,,,
2,SEC-01,seguridad,True,"[POL-SEC-003, INC-SEC-006]",,,,,
3,REM-01,trabajo remoto,True,[POL-REM-004],,,,,
4,CAP-01,capacitación,True,[POL-BEN-007],,,,,
5,NONE-01,sin evidencia,False,[],,,,,


## 9. De una interfaz de búsqueda a una capacidad seleccionable

Hasta acá respondimos **cómo recuperar mejor**. Ahora cambia la pregunta: **¿cómo exponemos esa búsqueda para que otra capa pueda decidir cuándo usarla?**

Llamamos a la capacidad `search_course()` para mostrar el cambio arquitectónico:

```text
RAG como aplicación completa
        ↓
RAG como capacidad invocable
```

En el **caso transversal de emails**, el mismo patrón representa:

```text
search_support_kb(query)
```

No son dos conceptos distintos. Es la misma abstracción aplicada a dos dominios diferentes.

Lo importante es que quien consume la capacidad no necesite conocer embeddings, chunks, BM25, RRF o reranking.


### 9.1 Definición operativa de Tool en C8

Una **Tool** es una capacidad delimitada que la aplicación puede poner a disposición del modelo para realizar una operación concreta.

En C8 todavía no implementamos el protocolo de ejecución. Nos alcanza con razonar sobre:

- **propósito**: qué resuelve;
- **entrada**: qué necesita;
- **resultado**: qué devuelve;
- **límites**: cuándo no corresponde usarla.

Para el caso de mails, la capacidad central será buscar evidencia en la base interna de soporte.


### 9.2 La descripción también es parte de la interfaz

Una descripción útil no dice solamente “busca cosas”.

```text
search_course(query)

Busca evidencia en el corpus interno.
Usar cuando la respuesta depende de conocimiento documentado.
No usar para cálculos ni para datos externos que cambian en tiempo real.
Devuelve evidencia relevante con fuente y metadata.
```

La implementación interna puede cambiar de Dense a Hybrid o incorporar reranking sin que cambie el contrato externo.

Más adelante, en el caso de soporte por email, aplicaremos exactamente el mismo patrón a otra base de conocimiento.


In [49]:
# Describimos capacidades como contratos: nombre, entrada, salida y criterio de uso.
# Más adelante el router leerá estas descripciones para decidir una ruta, pero no la ejecutará.
CAPABILITIES = [
    {
        "name": "search_course",
        "input": "query: str",
        "result": "evidencia interna relevante con fuente y metadata",
        "description": (
            "Busca evidencia en el corpus interno. "
            "Usar cuando la respuesta depende de políticas, procedimientos "
            "o conocimiento documentado. "
            "No usar para cálculos ni datos externos vigentes."
        ),
    },
    {
        "name": "calculator",
        "input": "expression: str",
        "result": "resultado numérico",
        "description": (
            "Resuelve operaciones matemáticas exactas. "
            "No usar para buscar información documental."
        ),
    },
    {
        "name": "external_lookup",
        "input": "query: str",
        "result": "dato vigente desde una fuente externa",
        "description": (
            "Consulta información externa que puede cambiar con el tiempo, "
            "como estado actual de un servicio o cotización. "
            "No usar para conocimiento estable del corpus interno."
        ),
    },
    {
        "name": "direct_answer",
        "input": "message: str",
        "result": "respuesta directa",
        "description": (
            "Responder sin herramienta cuando la tarea puede resolverse "
            "con el contexto conversacional disponible."
        ),
    },
]

show_table(
    CAPABILITIES,
    ["name", "input", "result", "description"],
)


,name,input,result,description
0,search_course,query: str,evidencia interna relevante con fuente y metadata,Busca evidencia en el corpus interno. Usar cua...
1,calculator,expression: str,resultado numérico,Resuelve operaciones matemáticas exactas. No u...
2,external_lookup,query: str,dato vigente desde una fuente externa,Consulta información externa que puede cambiar...
3,direct_answer,message: str,respuesta directa,Responder sin herramienta cuando la tarea pued...


## 10. Semantic Router / LLM Router: decidir antes de ejecutar

Hasta ahora el LLM se usó sobre todo para producir respuestas.

Ahora le asignamos primero una tarea más acotada:

> **interpretar la intención y seleccionar una capacidad.**

Entrada:

```text
consulta + descripciones de capacidades
```

Salida:

```json
{"route": "search_course"}
```

El router **no ejecuta** `search_course`, `calculator` ni `external_lookup`. Solo decide qué camino corresponde.


### 10.1 Structured Output para la decisión de ruteo

Reutilizamos la idea de C2: un resultado validado por Pydantic.

Las rutas permitidas son:

- `search_course`
- `calculator`
- `external_lookup`
- `direct_answer`

En smoke mode usamos un clasificador determinístico para que la notebook sea reproducible. Opcionalmente, podemos reemplazarlo por un LLM real con el mismo contrato.

El patrón **Semantic Router** no exige necesariamente un LLM: según el problema puede implementarse con reglas, un clasificador, embeddings o un modelo generativo. Acá usamos un LLM opcional porque permite reutilizar las **descripciones semánticas** y los Structured Outputs ya vistos.

> **Routing no elimina las alucinaciones.** Reduce el riesgo de elegir una capacidad incorrecta, pero el router puede clasificar mal y una capacidad puede devolver evidencia pobre. Por eso siguen siendo necesarias validación, evaluación y observabilidad.


In [50]:
%%writefile ai_agent_project/src/ai_agent_course/routing.py
from __future__ import annotations

import os
import re
from typing import Literal, Sequence

from pydantic import BaseModel, Field


RouteName = Literal[
    "search_course",
    "calculator",
    "external_lookup",
    "direct_answer",
]


class RouteDecision(BaseModel):
    """Contrato validado para la decisión del router.

    ``route`` limita las salidas posibles; ``confidence`` y ``reason`` agregan
    información útil para inspección. El objeto describe una decisión, no ejecuta
    la capacidad elegida.
    """

    route: RouteName
    confidence: float = Field(ge=0.0, le=1.0)
    reason: str = Field(min_length=3, max_length=180)



def build_router_prompt(
    user_message: str,
    capabilities: Sequence[dict],
) -> str:
    """Construye el prompt de clasificación a partir de capacidades declaradas.

    Args:
        user_message: Consulta que queremos enrutar.
        capabilities: Contratos con nombre y descripción de cada capacidad.

    Returns:
        Prompt textual que pide seleccionar exactamente una ruta sin ejecutarla.
    """
    lines = [
        "Clasificá la consulta en exactamente una capacidad.",
        "No ejecutes ninguna capacidad.",
        "",
        "CAPACIDADES:",
    ]

    # Las descripciones son parte del contrato semántico: explican al router
    # cuándo corresponde elegir cada capacidad.
    for capability in capabilities:
        lines.append(
            f"- {capability['name']}: {capability['description']}"
        )

    lines.extend([
        "",
        f"CONSULTA: {user_message}",
    ])

    return "\n".join(lines)


class RuleSemanticRouter:
    """Fallback determinístico para estudiar routing sin depender de un LLM.

    Reproduce el mismo contrato ``route(message, capabilities)`` para que el
    benchmark sea ejecutable offline. Sus reglas son didácticas, no generales.
    """

    name = "rule-router-smoke-only"

    def route(
        self,
        user_message: str,
        capabilities: Sequence[dict],
    ) -> RouteDecision:
        """Clasifica una consulta mediante reglas y devuelve una decisión.

        Args:
            user_message: Mensaje que se quiere enrutar.
            capabilities: Se recibe para mantener el mismo contrato que el router LLM.

        Returns:
            ``RouteDecision``. No invoca ninguna capacidad.
        """
        text = user_message.lower().strip()

        # El orden importa: primero detectamos necesidades que no debe resolver
        # el corpus (cálculo exacto o información vigente), y recién después KB.
        explicit_math = re.search(
            r"\d+(?:[\.,]\d+)?\s*[\+\-\*/]\s*\d+(?:[\.,]\d+)?",
            text,
        )
        calculation_markers = {
            "calculá",
            "calcula",
            "cuánto da",
            "cuanto da",
            "cuántas quedan",
            "cuantas quedan",
            "diferencia",
            "sumá",
            "suma",
            "multiplicá",
            "multiplica",
        }

        if explicit_math or (
            any(marker in text for marker in calculation_markers)
            and re.search(r"\d", text)
        ):
            return RouteDecision(
                route="calculator",
                confidence=0.96,
                reason="La consulta requiere una operación matemática exacta.",
            )

        current_markers = {
            "hoy",
            "actual",
            "ahora",
            "cotización",
            "tipo de cambio",
            "clima",
            "estado del servicio",
        }

        if any(marker in text for marker in current_markers):
            return RouteDecision(
                route="external_lookup",
                confidence=0.92,
                reason="La respuesta depende de información externa o vigente.",
            )

        corpus_markers = {
            "curso",
            "clase",
            "mrr",
            "rrf",
            "bm25",
            "vacaciones",
            "reintegro",
            "credencial",
            "contraseña",
            "política",
            "capacitación",
            "trabajo remoto",
            "factura",
            "portal",
            "integración",
            "api",
            "sso",
            "perfil",
        }

        if any(marker in text for marker in corpus_markers):
            return RouteDecision(
                route="search_course",
                confidence=0.90,
                reason="La respuesta depende del corpus interno disponible.",
            )

        return RouteDecision(
            route="direct_answer",
            confidence=0.70,
            reason="No se detecta necesidad de cálculo, corpus ni dato externo.",
        )


class GeminiSemanticRouter:
    """Router opcional que usa Structured Output de Gemini para clasificar.

    Mantiene el mismo contrato que el fallback por reglas, lo que permite cambiar
    implementación sin modificar el benchmark ni el código consumidor.
    """

    def __init__(
        self,
        api_key: str,
        model: str = "gemini-3.1-flash-lite",
    ):
        """Inicializa el cliente de Gemini.

        Args:
            api_key: Credencial del proveedor.
            model: Modelo que realizará la clasificación.

        Returns:
            None. Configura cliente, modelo y nombre del router.
        """
        if not api_key:
            raise ValueError("API key vacía")

        from google import genai

        self.client = genai.Client(api_key=api_key)
        self.model = model
        self.name = model

    def route(
        self,
        user_message: str,
        capabilities: Sequence[dict],
    ) -> RouteDecision:
        """Pide al modelo una decisión estructurada de routing.

        Args:
            user_message: Consulta a clasificar.
            capabilities: Capacidades disponibles y sus descripciones.

        Returns:
            ``RouteDecision`` validada por Pydantic.

        El modelo solo selecciona una capacidad; la ejecución queda fuera de C8.
        """
        from google.genai import types

        prompt = build_router_prompt(
            user_message,
            capabilities,
        )

        # response_schema obliga a que la salida respete el mismo contrato
        # estructurado que usamos en el fallback determinístico.
        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.0,
                response_mime_type="application/json",
                response_schema=RouteDecision,
            ),
        )

        # Preferimos la respuesta ya parseada por el SDK. Si no está disponible,
        # validamos manualmente el JSON textual con el mismo modelo Pydantic.
        if getattr(response, "parsed", None) is not None:
            return response.parsed

        return RouteDecision.model_validate_json(
            response.text or "{}"
        )


Overwriting ai_agent_project/src/ai_agent_course/routing.py


In [1]:
# Por defecto usamos el router determinístico para mantener la notebook reproducible.
# USE_REAL_ROUTER=1 cambia solo la implementación, no el contrato ni el benchmark.
import ai_agent_course.routing as routing_module
importlib.reload(routing_module)

router = routing_module.RuleSemanticRouter()

USE_REAL_ROUTER = True

# El router real requiere credenciales explícitas; no inventamos ni ocultamos ese requisito.
if USE_REAL_ROUTER:
    api_key = (
        ""
        or os.getenv("GOOGLE_API_KEY")
    )

    if not api_key:
        raise RuntimeError(
            "USE_REAL_ROUTER=1 requiere GEMINI_API_KEY o GOOGLE_API_KEY."
        )

    router = routing_module.GeminiSemanticRouter(
        api_key=api_key,
        model=os.getenv(
            "GEMINI_MODEL",
            "gemini-3.1-flash-lite",
        ),
    )

print("Router activo:", router.name)


ModuleNotFoundError: No module named 'ai_agent_course'

In [59]:
# Golden específico para ROUTING.
# No mide retrieval: mide si elegimos correctamente la capacidad.
ROUTING_GOLDEN = [
    {
        "case_id": "R-01",
        "message": "¿Con cuánta anticipación pido vacaciones?",
        "expected": "search_course",
        "kind": "conocimiento interno",
    },
    {
        "case_id": "R-02",
        "message": "Expuse mi contraseña. ¿Qué política debo seguir?",
        "expected": "search_course",
        "kind": "conocimiento interno",
    },
    {
        "case_id": "R-03",
        "message": thread_to_query(
            next(
                thread
                for thread in THREADS
                if thread["thread_id"] == "thr_001"
            )
        ),
        "expected": "search_course",
        "kind": "email / conocimiento interno",
    },
    {
        "case_id": "R-04",
        "message": thread_to_query(
            next(
                thread
                for thread in THREADS
                if thread["thread_id"] == "thr_003"
            )
        ),
        "expected": "search_course",
        "kind": "email / conocimiento interno",
    },
    {
        "case_id": "R-05",
        "message": "Calculá 21 - 8.",
        "expected": "calculator",
        "kind": "cálculo",
    },
    {
        "case_id": "R-06",
        "message": "Tengo 21 licencias y usé 8. ¿Cuántas quedan?",
        "expected": "calculator",
        "kind": "cálculo",
    },
    {
        "case_id": "R-07",
        "message": "¿Cuál es el estado actual del portal?",
        "expected": "external_lookup",
        "kind": "dato vigente",
    },
    {
        "case_id": "R-08",
        "message": "¿Cuál es la cotización hoy?",
        "expected": "external_lookup",
        "kind": "dato vigente",
    },
    {
        "case_id": "R-09",
        "message": "Reescribí esta frase con un tono más claro.",
        "expected": "direct_answer",
        "kind": "respuesta directa",
    },
    {
        "case_id": "R-10",
        "message": "Dame tres títulos alternativos para este texto.",
        "expected": "direct_answer",
        "kind": "respuesta directa",
    },
]

# Evaluamos routing con un golden independiente del golden de retrieval.
# Así un error de clasificación no se confunde con un error del buscador.
router_rows = []

for case in ROUTING_GOLDEN:
    # El benchmark observa solo la decisión; ninguna Tool se ejecuta en esta etapa.
    decision = router.route(
        case["message"],
        CAPABILITIES,
    )

    router_rows.append({
        "case_id": case["case_id"],
        "kind": case["kind"],
        "message": case["message"],
        "expected": case["expected"],
        "route": decision.route,
        "confidence": decision.confidence,
        "reason": decision.reason,
        "match": decision.route == case["expected"],
    })

# Accuracy global responde "cuántas rutas acertó"; el corte por tipo ayuda
# a detectar si una clase de intención específica está fallando.
routing_accuracy = round(
    sum(row["match"] for row in router_rows)
    / len(router_rows),
    3,
)

routing_by_kind = []

for kind in sorted(
    {row["kind"] for row in router_rows}
):
    rows = [
        row
        for row in router_rows
        if row["kind"] == kind
    ]

    routing_by_kind.append({
        "kind": kind,
        "cases": len(rows),
        "accuracy": round(
            sum(row["match"] for row in rows)
            / len(rows),
            3,
        ),
    })

show_table(router_rows)
print("Routing accuracy:", routing_accuracy)
show_table(routing_by_kind)


,case_id,kind,message,expected,route,confidence,reason,match
0,R-01,conocimiento interno,¿Con cuánta anticipación pido vacaciones?,search_course,search_course,1.00,La consulta requiere acceder a políticas inter...,True
1,R-02,conocimiento interno,Expuse mi contraseña. ¿Qué política debo seguir?,search_course,search_course,1.00,La consulta requiere acceder a políticas inter...,True
2,R-03,email / conocimiento interno,Error al cargar facturas en el portal No podem...,search_course,search_course,0.95,El usuario reporta un error técnico en un port...,True
3,R-04,email / conocimiento interno,Pedido de integración con sistema contable Que...,search_course,search_course,0.95,La consulta busca información sobre capacidade...,True
4,R-05,cálculo,Calculá 21 - 8.,calculator,calculator,1.00,La consulta solicita explícitamente realizar u...,True
5,R-06,cálculo,Tengo 21 licencias y usé 8. ¿Cuántas quedan?,calculator,calculator,1.00,La consulta requiere realizar una operación ar...,True
6,R-07,dato vigente,¿Cuál es el estado actual del portal?,external_lookup,external_lookup,1.00,La consulta requiere verificar el estado actua...,True
7,R-08,dato vigente,¿Cuál es la cotización hoy?,external_lookup,external_lookup,1.00,"La consulta solicita una cotización actual, la...",True
8,R-09,respuesta directa,Reescribí esta frase con un tono más claro.,direct_answer,direct_answer,1.00,La tarea de reescribir una frase es una activi...,True
9,R-10,respuesta directa,Dame tres títulos alternativos para este texto.,direct_answer,direct_answer,1.00,La tarea de generar títulos alternativos para ...,True


Routing accuracy: 1.0


,kind,cases,accuracy
0,conocimiento interno,2,1.0
1,cálculo,2,1.0
2,dato vigente,2,1.0
3,email / conocimiento interno,2,1.0
4,respuesta directa,2,1.0


### 10.2 Router ≠ Agent

Este punto marca exactamente el límite entre C8 y C9.

Un **router** toma una decisión de clasificación:

```text
consulta → route
```

Un **agente** puede entrar en un ciclo:

```text
decisión → acción → observación → nueva decisión
```

En esta notebook nos detenemos después de la primera decisión. No hay Tool Registry, no hay Function Calling y no ejecutamos la capacidad elegida desde el router.


In [60]:
# Verificamos que el router SOLO produzca decisiones y medimos el contrato.
# La validación comprueba dos propiedades: rutas dentro del contrato
# y cantidad suficiente de casos para que el smoke benchmark sea útil.
allowed_routes = {
    "search_course",
    "calculator",
    "external_lookup",
    "direct_answer",
}

assert all(
    row["route"] in allowed_routes
    for row in router_rows
)
assert len(ROUTING_GOLDEN) >= 8

print({
    "routing_cases": len(ROUTING_GOLDEN),
    "accuracy": routing_accuracy,
    "executes_tools": False,
})

print(
    "✓ C8 termina en selección de capacidad. "
    "La ejecución controlada pertenece a C9."
)


{'routing_cases': 10, 'accuracy': 1.0, 'executes_tools': False}
✓ C8 termina en selección de capacidad. La ejecución controlada pertenece a C9.


## 11. Aplicar el patrón al caso transversal de mails

Con el benchmark técnico cerrado y el router evaluado, volvemos al caso de negocio.

Ahora construimos una **segunda instancia** del mismo patrón de búsqueda, esta vez sobre la KB de soporte que C7 dejó preparada.

Esto muestra una idea importante:

> BM25, RRF, reranking y una interfaz estable no pertenecen al dominio “curso”.  
> Son piezas reutilizables que pueden aplicarse a otro corpus.

```text
benchmark C6
search_course(query)
        ↓
misma arquitectura

caso de emails
search_support_kb(query)
```

El benchmark técnico y el caso transversal quedan conectados, pero no mezclados.


In [61]:
# Índice separado para la KB de soporte del caso de emails.
# Reindexamos la KB de soporte por separado para no mezclar dominios.
# La arquitectura es la misma; cambian únicamente corpus y casos de evaluación.
support_dense_retriever = build_retriever(
    use_qdrant=False,
    path=DATA_DIR / "class08_support",
    embedding_provider=embedding_provider,
)
support_dense_retriever.index(SUPPORT_CHUNKS)

support_search_service = course_search_module.CourseSearchService(
    dense_retriever=support_dense_retriever,
    chunks=SUPPORT_CHUNKS,
    reranker=reranker,
    strategy="hybrid_rerank",
    candidate_k=8,
)

def search_support_kb(
    query: str,
    top_k: int = 3,
) -> list[dict]:
    """Busca evidencia de soporte reutilizando la arquitectura de C8.

    Args:
        query: Consulta construida desde un hilo de email.
        top_k: Cantidad máxima de evidencias a devolver.

    Returns:
        Resultados serializados con documento, chunk, texto, fuente y metadata.

    Esta función demuestra reutilización: cambia el corpus, no el patrón
    Dense + BM25 + RRF + reranking.
    """
    results, _trace = support_search_service.search(
        query,
        top_k=top_k,
    )
    return _serialize_results(results)

email_case_rows = []

# El flujo del caso transversal queda explícito: primero routing y, solo si
# corresponde conocimiento interno, retrieval sobre la KB de soporte.
for case in EMAIL_CASES:
    decision = router.route(
        case["query"],
        CAPABILITIES,
    )

    evidence = []

    # En C8 ejecutamos esta rama manualmente para mostrar la frontera.
    # La ejecución automática de Tools queda reservada para C9.
    if decision.route == "search_course":
        evidence = search_support_kb(
            case["query"],
            top_k=3,
        )

    retrieved_ids = [
        item["document_id"]
        for item in evidence
    ]

    email_case_rows.append({
        "thread_id": case["thread_id"],
        "subject": case["subject"],
        "route": decision.route,
        "expected_route": "search_course",
        "routing_match": (
            decision.route == "search_course"
        ),
        "expected_document": case[
            "relevant_document_ids"
        ][0],
        "top_document_ids": retrieved_ids,
        "retrieval_hit@3": (
            case["relevant_document_ids"][0]
            in retrieved_ids
        ),
    })

show_table(email_case_rows)


,thread_id,subject,route,expected_route,routing_match,expected_document,top_document_ids,retrieval_hit@3
0,thr_001,Error al cargar facturas en el portal,search_course,search_course,True,SUP-FACTURAS,"[SUP-FACTURAS, SUP-VALIDACION, SUP-API]",True
1,thr_002,Consulta sobre cambio de contraseña,search_course,search_course,True,SUP-PASSWORD,"[SUP-PASSWORD, SUP-FACTURAS, SUP-API]",True
2,thr_003,Pedido de integración con sistema contable,search_course,search_course,True,SUP-API,"[SUP-API, SUP-FACTURAS, SUP-VALIDACION]",True


## 12. Checks de cierre

Antes de generar el reporte verificamos tres niveles distintos:

### A. Benchmark técnico

- el corpus y el golden provienen del handoff de C6;
- mantenemos los 6 casos, incluido el negativo;
- comparamos las tres estrategias sobre exactamente las mismas queries;
- conservamos Precision / Recall / Hit Rate para `k=1,3,5`;
- conservamos MRR y latencia.

### B. Routing

- existe un golden separado;
- el router devuelve una ruta validada;
- routing accuracy se mide independientemente de retrieval;
- el router no ejecuta capacidades.

### C. Caso transversal

- reutilizamos los threads de C7;
- la KB de soporte usa la misma arquitectura Hybrid + Rerank;
- cada mail conserva trazabilidad hacia el documento esperado;
- Function Calling y Agent Loop siguen fuera de C8.


In [62]:
# Checks de invariantes: validan continuidad, cobertura del benchmark y contratos
# entre etapas. No prueban calidad absoluta, pero detectan roturas estructurales.
assert len(RETRIEVAL_GOLDEN) == len(
    class06_report["handoff"]["golden_dataset"]
)
assert len(RETRIEVAL_GOLDEN) >= 6
assert any(
    not case["expects_evidence"]
    for case in RETRIEVAL_GOLDEN
)

assert {
    row["strategy"]
    for row in comparison_rows
} == set(STRATEGIES)

assert {
    row["k"]
    for row in aggregate_rows
} == set(K_VALUES)

assert all(
    metric in comparison_rows[0]
    for metric in [
        "precision@1",
        "recall@1",
        "hit@1",
        "precision@3",
        "recall@3",
        "hit@3",
        "precision@5",
        "recall@5",
        "hit@5",
        "mrr",
        "latency_ms",
    ]
)

# RRF no debe duplicar un mismo chunk en el ranking fusionado.
fused_chunk_ids = [
    row.result.chunk_id
    for row in hybrid_rows
]
assert len(fused_chunk_ids) == len(
    set(fused_chunk_ids)
)

# El reranker solo puede reordenar candidatos que estaban en el candidate set.
assert reranked_sample
assert all(
    any(
        candidate.result.chunk_id
        == ranked.result.chunk_id
        for candidate in hybrid_rows
    )
    for ranked in reranked_sample
)

# Routing se valida contra un espacio cerrado de rutas permitidas.
assert len(ROUTING_GOLDEN) >= 8
assert all(
    row["route"] in {
        "search_course",
        "calculator",
        "external_lookup",
        "direct_answer",
    }
    for row in router_rows
)

assert THREADS
assert EMAIL_CASES

print(
    "✓ C8 conserva el benchmark de C6, "
    "agrega routing y aplica el patrón al caso de emails."
)


✓ C8 conserva el benchmark de C6, agrega routing y aplica el patrón al caso de emails.


## 13. Reporte final de evidencia

El reporte de C8 ya no mezcla datasets con objetivos diferentes.

Guarda por separado:

1. **retrieval benchmark** heredado de C6;
2. comparación Dense vs Hybrid+RRF vs Hybrid+Rerank;
3. control negativo sin evidencia;
4. orden del contexto e intuición previa a Context Precision;
5. **routing benchmark**;
6. aplicación al caso transversal de mails;
7. handoff técnico hacia C9.

No se guarda un “ganador universal”.  
Se guarda evidencia para justificar cuándo una estrategia agrega valor.


In [63]:
# El reporte consolida evidencia y decisiones para que C9 pueda reutilizar
# resultados sin depender del estado en memoria de esta notebook.
report = {
    "class": 8,

    "case_study": {
        "name": "agente de gestión asistida de emails",
        "role_in_class": (
            "capa de aplicación y puente hacia Tools; "
            "no reemplaza el benchmark técnico"
        ),
        "threads": [
            {
                "thread_id": thread["thread_id"],
                "subject": thread["subject"],
            }
            for thread in THREADS
        ],
        "email_results": email_case_rows,
    },

    # Benchmark técnico: conserva dataset, resultados, métricas y diagnósticos.
    "retrieval_benchmark": {
        "source": "class06_report.json / handoff",
        "golden_dataset": RETRIEVAL_GOLDEN,
        "chunks": len(CHUNKS),
        "embedding_model": embedding_provider.model_name,
        "strategies": STRATEGIES,
        "per_case": comparison_rows,
        "aggregate_by_k": aggregate_rows,
        "mrr_and_latency": mrr_latency_summary,
        "diagnostics": diagnostic_rows,
        "no_evidence": {
            "case": negative_case,
            "results": negative_rows,
        },
    },

    "search_course": {
        "interface": (
            "search_course(query: str, top_k: int = 3) -> list[dict]"
        ),
        "default_strategy": search_service.strategy,
        "candidate_k": search_service.candidate_k,
        "bm25_backend": bm25.backend_name,
        "reranker": reranker.name,
    },

    "context_precision_intuition": {
        "case_id": focus_case["case_id"],
        "hybrid_rrf": relevance_trace(
            hybrid_focus,
            focus_case["relevant_document_ids"],
            top_k=5,
        ),
        "hybrid_rerank": relevance_trace(
            rerank_focus,
            focus_case["relevant_document_ids"],
            top_k=5,
        ),
        "formal_framework_used": False,
    },

    # Benchmark de decisión: se mantiene separado de retrieval por diseño.
    "routing_benchmark": {
        "router": router.name,
        "golden": ROUTING_GOLDEN,
        "results": router_rows,
        "accuracy": routing_accuracy,
        "accuracy_by_kind": routing_by_kind,
        "executes_tools": False,
    },

    "scope": {
        "ragas": "deferred_to_llmops",
        "llm_as_judge": "deferred_to_llmops",
        "function_calling": "next_class",
        "agent_loop": "next_class",
    },

    "continuity": {
        "increment": (
            "benchmark RAG avanzado + BM25 + RRF + reranking + "
            "routing + aplicación al caso de emails"
        ),
        "same_email_fixture_for_next_class": (
            "src/ai_agent_course/support_case.py"
        ),
        "modules": [
            "src/ai_agent_course/ranking.py",
            "src/ai_agent_course/reranker.py",
            "src/ai_agent_course/course_search.py",
            "src/ai_agent_course/routing.py",
        ],
        "next_class": 9,
    },
}

# Persistimos un handoff versionable y legible fuera del notebook.
report_path = ARTIFACTS_DIR / "class08_report.json"

write_file(
    report_path,
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2,
    ),
)

print(
    "✓ Reporte Clase 8 generado:",
    report_path.relative_to(PROJECT_DIR),
)

{
    "class": 8,
    "retrieval_cases": len(RETRIEVAL_GOLDEN),
    "routing_cases": len(ROUTING_GOLDEN),
    "email_cases": len(EMAIL_CASES),
    "report": str(
        report_path.relative_to(PROJECT_DIR)
    ),
    "next_class": 9,
}


✓ artifacts/class08_report.json
✓ Reporte Clase 8 generado: artifacts/class08_report.json


{'class': 8,
 'retrieval_cases': 6,
 'routing_cases': 10,
 'email_cases': 3,
 'report': 'artifacts/class08_report.json',
 'next_class': 9}

## 14. Continuidad del proyecto

La secuencia completa queda más clara:

```text
C6 — Retrieval evaluable
corpus
  + ground truth
  + Precision / Recall / Hit Rate / MRR / latencia
          ↓

C7 — RAG end-to-end
retrieval → contexto → generación → trazabilidad / abstención
          ↓

C8 — RAG avanzado + selección de capacidad
mismo benchmark de C6
  ├─ Dense
  ├─ Hybrid + RRF
  └─ Hybrid + Rerank
          ↓
diagnóstico por síntoma
          ↓
RAG Tournament
          ↓
interfaz estable + descripción de capacidades
          ↓
Semantic Router
          ↓
aplicación al caso de emails
          ↓

C9 — Tools y Agentic RAG
modelo decide Tool + argumentos
          ↓
validación / permisos
          ↓
ejecución
          ↓
observación
          ↓
nueva decisión
```

C8 deja reutilizables:

```text
ranking.py       → BM25 + RRF
reranker.py      → Cross-Encoder / fallback explícito
course_search.py → interfaz estable de búsqueda
routing.py       → decisión semántica entre capacidades
support_case.py  → mismos mails que C9 seguirá utilizando
```

Y, sobre todo, conserva dos activos importantes:

- el **ground truth histórico** para seguir midiendo sin mover el arco;
- el **caso transversal de emails** para que el proyecto siga teniendo una historia de negocio.

**Pregunta de salida:** si hoy podemos medir qué estrategia recupera mejor, elegir una capacidad y aplicarla a un mail, ¿cómo hacemos en C9 para que el modelo declare una Tool, proponga argumentos válidos, la ejecute de forma controlada y observe el resultado para decidir otra vez?
